<a href="https://colab.research.google.com/github/E1000io-bit/agente-agro/blob/main/AgroAgente_TP2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AgroAgente: agente de consultas fitosanitarias

---

FCEFyN - UNC - Cátedra de Inteligencia Artificial · Trabajo de laboratorio 2

---

Agente que responde consultas por texto sobre plagas, enfermedades y malezas de maíz, soja, trigo y maní en Argentina. Usa como base de conocimiento las fichas oficiales de SINAVIO (SENASA) y los informes de la Red Nacional de Monitoreo de la chicharrita del maíz, consulta el pronóstico del lugar y cita la fuente. No receta productos ni dosis.
> Sobre el modelo y los datos: este agente no entrena ningún modelo. El LLM (`Qwen/Qwen3-4B-Instruct-2507`) ya viene entrenado y corre dentro de Colab, en la GPU. Los datos de la carpeta del Drive son la base de conocimiento que el agente consulta en cada pregunta y los sets de evaluación. A partir de ellos sólo se construyen los índices de búsqueda (BM25 y embeddings).

## 1. Instalación de dependencias

In [ ]:
!pip install -q langchain langgraph transformers accelerate requests beautifulsoup4 pypdf

In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
import os, re, json, math, time, unicodedata, csv, uuid, warnings
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import requests
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 2. Datos

Los datos están en la carpeta **`AgroAgente_TP2_datos`** del Drive del grupo

| Carpeta | Contenido |
|---|---|
| `corpus/sinavimo/` | 70 fichas oficiales de SINAVIMO (SENASA) |
| `corpus/chicharrita/` | 10 informes de la Red Nacional de Monitoreo de *Dalbulus maidis* (N° 42–50 y 52, hasta el 20/10/2026) |
| `plagas_objetivo.csv` | Las 70 plagas elegidas y sus nombres usados en Argentina |
| `evaluacion/` | Preguntas de evaluación (escritas a mano: **sólo están en el Drive**) |
| `resultados/` | Acá se guardan los resultados que genera esta notebook |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATA_DIR = Path(os.environ.get("AGRO_DATA_DIR", "/content/drive/MyDrive/Inteligencia Artificial/AgroAgente_TP2_datos"))

if DATA_DIR.exists():
    fichas = sorted((DATA_DIR / "corpus" / "sinavimo").glob("*.md"))
    informes = sorted((DATA_DIR / "corpus" / "chicharrita").glob("*.md"))
    print(f"Datos en {DATA_DIR}: {len(fichas)} fichas, {len(informes)} informes de chicharrita")
else:
    print(f"No encontré {DATA_DIR}.\n"
          "Opción A: subí la carpeta AgroAgente_TP2_datos a 'Mi unidad' y volvé a correr esta celda.\n"
          "Opción B: corré la celda siguiente (RECONSTRUIR = True) para descargar los datos de las fuentes oficiales.")

Datos en /content/drive/MyDrive/Inteligencia Artificial/AgroAgente_TP2_datos: 70 fichas, 10 informes de chicharrita


### 2 Descargar los datos de las fuentes oficiales

En caso de no tener acceso al dataset del drive ponerlo en true para que asi se descargue de fuentes oficiales ficheros que se pueden usar para testear el codigo.
Se encarga de bajar las 70 fichas de SINAVIMO (Senasa) y los últimos informes de chicharrita. Tarda unos minutos y espera entre pedidos para no cargar los servidores. Sirve para reproducir el armado de los datos o si no hay acceso al Drive.

In [ ]:
RECONSTRUIR = not DATA_DIR.exists()     # poner True para forzar la descarga

PLAGAS_CSV = """slug,cultivos,grupo,alias_ar
dalbulus-maidis,maiz,insecto,Chicharrita del maíz
spiroplasma-kunkelii,maiz,bacteria,Achaparramiento del maíz;Corn stunt
maize-bushy-stunt-phytoplasma,maiz,fitoplasma,Achaparramiento por fitoplasma
maize-rayado-fino-virus,maiz,virus,Rayado fino
mal-de-rio-cuarto-virus,maiz,virus,Mal de Río Cuarto
delphacodes-kuscheli,maiz;trigo,insecto,Delfácido
spodoptera-frugiperda,maiz;soja,insecto,Cogollero;Gusano cogollero
diatraea-saccharalis,maiz,insecto,Barrenador del tallo
helicoverpa-zea,maiz;soja,insecto,Isoca de la espiga
elasmopalpus-lignosellus,maiz;soja;mani,insecto,Barrenador menor del tallo
puccinia-sorghi,maiz,hongo,Roya común
puccinia-polysora,maiz,hongo,Roya polysora
exserohilum-turcicum,maiz,hongo,Tizón foliar;Tizón común
cercospora-zeae-maydis,maiz,hongo,Mancha gris
kabatiella-zeae,maiz,hongo,Mancha ocular
ustilago-maydis,maiz,hongo,Carbón común
stenocarpella-maydis,maiz,hongo,Podredumbre de tallo y espiga
maize-dwarf-mosaic-virus,maiz,virus,Mosaico enanizante
fusarium-graminearum,trigo;maiz,hongo,Fusariosis de la espiga;Golpe blanco
rachiplusia-nu,soja,insecto,Isoca medidora
anticarsia-gemmatalis,soja,insecto,Oruga de las leguminosas
chrysodeixis-includens,soja,insecto,Falsa medidora
helicoverpa-gelotopoeon,soja;mani,insecto,Oruga bolillera;Isoca bolillera
spodoptera-cosmioides,soja,insecto,
nezara-viridula,soja,insecto,Chinche verde
piezodorus-guildinii,soja,insecto,Chinche de la alfalfa
dichelops-furcatus,soja;maiz;trigo,insecto,Chinche de los cuernos
tetranychus-urticae,soja,acaro,Arañuela roja;Arañuela de las dos manchas
caliothrips-phaseoli,soja,insecto,Trips
rhyssomatus-subtilis,soja,insecto,Picudo negro
sternechus-subsignatus,soja,insecto,Picudo de la soja
spissistilus-festinus,soja,insecto,Membrácido
phakopsora-pachyrhizi,soja,hongo,Roya asiática
cercospora-sojina,soja,hongo,Mancha ojo de rana
cercospora-kikuchii,soja,hongo,Tizón de la hoja;Mancha púrpura de la semilla
septoria-glycines,soja,hongo,Mancha marrón
macrophomina-phaseolina,soja;maiz;mani,hongo,Podredumbre carbonosa
sclerotinia-sclerotiorum,soja;mani,hongo,Podredumbre húmeda del tallo;Moho blanco
diaporthe-phaseolorum-var-sojae,soja,hongo,Tizón de la vaina y tallo
diaporthe-caulivora,soja,hongo,Cancro del tallo
heterodera-glycines,soja,nematodo,Nematodo del quiste
puccinia-recondita,trigo,hongo,Roya de la hoja;Roya anaranjada
puccinia-striiformis,trigo,hongo,Roya amarilla
puccinia-graminis,trigo,hongo,Roya del tallo;Roya negra
zymoseptoria-tritici,trigo,hongo,Mancha de la hoja;Septoriosis
pyrenophora-tritici-repentis,trigo,hongo,Mancha amarilla;Mancha tostada
bipolaris-sorokiniana,trigo,hongo,Mancha borrosa
blumeria-graminis-tritici,trigo,hongo,Oídio
ustilago-tritici,trigo,hongo,Carbón volador
schizaphis-graminum,trigo,insecto,Pulgón verde de los cereales
rhopalosiphum-padi,trigo;maiz,insecto,Pulgón de la avena
sitobion-avenae,trigo,insecto,Pulgón de la espiga
metopolophium-dirhodum,trigo,insecto,Pulgón amarillo de los cereales
diloboderus-abderus,varios,insecto,Gusano blanco;Bicho torito
agrotis-ipsilon,varios,insecto,Gusano cortador;Gusano grasiento
dichroplus-maculipennis,varios,insecto,Tucura
schistocerca-cancellata,varios,insecto,Langosta
thecaphora-frezzii,mani,hongo,Carbón del maní
sclerotinia-minor,mani,hongo,Tizón del maní
puccinia-arachidis,mani,hongo,Roya del maní
amaranthus-hybridus-subsp-quitensis,soja;maiz,maleza,Yuyo colorado;Ataco
amaranthus-palmeri,soja;maiz,maleza,Yuyo colorado;Palmeri
sorghum-halepense,soja;maiz,maleza,Sorgo de Alepo
conyza-bonariensis,soja;maiz;trigo,maleza,Rama negra
eleusine-indica,soja;maiz,maleza,Pie de gallina;Grama carraspera
lolium-multiflorum,trigo;soja,maleza,Raigrás;Ryegrass
echinochloa-colona,soja;maiz,maleza,Capín;Arrocillo
chloris-virgata,soja;maiz,maleza,Plumerito;Cloris
avena-fatua,trigo,maleza,Avena negra;Avena guacha
brassica-rapa,trigo,maleza,Nabo;Nabolsa
"""

def _limpiar(t):
    t = t.replace("\xa0", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = "\n".join(l.strip() for l in t.splitlines())
    return re.sub(r"\n{3,}", "\n\n", t).strip()

def descargar_ficha_sinavimo(sesion, slug, meta):
    """Baja una ficha de https://www.sinavimo.gob.ar/plaga/<slug> y la convierte a Markdown con encabezado YAML."""
    from bs4 import BeautifulSoup
    url = f"https://www.sinavimo.gob.ar/plaga/{slug}"
    soup = BeautifulSoup(sesion.get(url, timeout=60).text, "html.parser")
    art = soup.find("article") or soup
    titulo = _limpiar((soup.find("h1") or soup.title).get_text(" "))
    secciones = {}
    for label in art.select(".field__label"):
        nombre = _limpiar(label.get_text(" ")).rstrip(":").strip()
        if not nombre or nombre in secciones:
            continue
        cont = label.parent
        items = list(dict.fromkeys(_limpiar(li.get_text(" ")) for li in cont.select(".field__item, li")))
        items = [i for i in items if i]
        if items and nombre.lower() not in {"descripción biológica", "signos y síntomas / daños", "condiciones predisponentes"}:
            secciones[nombre] = "\n".join(f"- {i}" for i in items)
        else:
            copia = BeautifulSoup(str(cont), "html.parser")
            for l in copia.select(".field__label"):
                l.decompose()
            for e in copia.find_all(["br", "p", "li"]):
                e.append("\n")
            secciones[nombre] = _limpiar(copia.get_text(" "))
    lista = lambda s: [l.lstrip("- ").strip() for l in (s or "").splitlines() if l.strip()]
    q = lambda s: '"' + s.replace('"', '\\"') + '"'
    fm = ["---", f"slug: {slug}", f"titulo: {q(titulo)}",
          "nombres_vulgares: [" + ", ".join(q(n) for n in lista(secciones.get("Nombre vulgar"))) + "]",
          f"tipo_plaga: {q(' / '.join(lista(secciones.get('Tipo de plaga'))))}",
          f"condicion: {q(' / '.join(lista(secciones.get('Condición'))))}",
          f"cultivos_tp: [{', '.join(meta['cultivos'].split(';'))}]", f"grupo: {meta['grupo']}",
          "fuente: SINAVIMO - SENASA", f"url: {url}", f"fecha_descarga: {datetime.now():%Y-%m-%d}", "---", "", f"# {titulo}", "", ""]
    cuerpo = "\n".join(f"## {k}\n\n{v}\n" for k, v in secciones.items() if v)
    return "\n".join(fm) + cuerpo + f"_Fuente: [SINAVIMO – {titulo}]({url})_\n"

def descargar_informes_chicharrita(sesion, destino, n=10):
    """Baja los n informes más recientes de la Red Nacional de Monitoreo (listado de MAIZAR) y extrae el texto."""
    import html as _html
    from urllib.parse import quote, urljoin
    from pypdf import PdfReader
    lista = "https://www.maizar.org.ar/vertodas_area.php?id=39"
    r = sesion.get(lista, timeout=60); r.encoding = r.apparent_encoding or "latin-1"
    items = []
    for m in re.finditer(r'href="(vertext\.php\?id=\d+)"[^>]*>(.*?)</a>', r.text, re.S):
        tit = re.sub(r"\s+", " ", _html.unescape(re.sub(r"<[^>]+>", " ", m.group(2)))).strip()
        num = re.search(r"(\d+)\s*°", tit)
        if "Informe" in tit and num and all(i["pagina"] != m.group(1) for i in items):
            items.append({"pagina": urljoin(lista, m.group(1)), "titulo": tit, "numero": int(num.group(1))})
    fechas = re.findall(r"(\d{2}/\d{2}/\d{4})", re.sub(r"<[^>]+>", " ", r.text))
    for it, f in zip(items, fechas):
        it["fecha"] = datetime.strptime(f, "%d/%m/%Y").date().isoformat()
    (destino / "pdf").mkdir(parents=True, exist_ok=True)
    for it in items[:n]:
        p = sesion.get(it["pagina"], timeout=60); p.encoding = p.apparent_encoding or "latin-1"
        m = re.search(r'href="([^"]+\.pdf)"', p.text, re.I)
        if not m:
            continue
        url = quote(urljoin(it["pagina"], _html.unescape(m.group(1))), safe=":/%?=&")
        pdf = destino / "pdf" / f"informe-{it['numero']:03d}.pdf"
        pdf.write_bytes(sesion.get(url, timeout=120).content)
        texto = "\n\n".join(f"<!-- página {i} -->\n{(pg.extract_text() or '').strip()}"
                            for i, pg in enumerate(PdfReader(str(pdf)).pages, 1))
        fm = (f"---\nslug: chicharrita-informe-{it['numero']:03d}\ntitulo: \"{it['titulo']}\"\nnumero: {it['numero']}\n"
              f"fecha: {it.get('fecha', '')}\ncultivos_tp: [maiz]\nfuente: Red Nacional de Monitoreo (MAIZAR / INTA)\nurl: {url}\n---\n\n")
        (destino / f"informe-{it['numero']:03d}.md").write_text(fm + f"# {it['titulo']}\n\n{texto}\n", encoding="utf-8")
        print("  ok", pdf.name, it.get("fecha"))
        time.sleep(1.5)

if RECONSTRUIR:
    DATA_DIR = Path("/content/AgroAgente_TP2_datos") if Path("/content").exists() else Path("AgroAgente_TP2_datos")
    (DATA_DIR / "corpus" / "sinavimo").mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "plagas_objetivo.csv").write_text(PLAGAS_CSV, encoding="utf-8")
    sesion = requests.Session(); sesion.headers["User-Agent"] = "TP-IA-agente-agro/1.0 (uso academico)"
    for fila in csv.DictReader(PLAGAS_CSV.splitlines()):
        destino = DATA_DIR / "corpus" / "sinavimo" / f"{fila['slug']}.md"
        if not destino.exists():
            destino.write_text(descargar_ficha_sinavimo(sesion, fila["slug"], fila), encoding="utf-8")
            time.sleep(1.5)
    print("Fichas:", len(list((DATA_DIR / "corpus" / "sinavimo").glob("*.md"))))
    descargar_informes_chicharrita(sesion, DATA_DIR / "corpus" / "chicharrita", n=10)
    print("Datos reconstruidos en", DATA_DIR)
else:
    print("Uso los datos del Drive:", DATA_DIR)

Uso los datos del Drive: /content/drive/MyDrive/Inteligencia Artificial/AgroAgente_TP2_datos


## 3. Base de conocimiento (RAG)

Recuperación de los fragmentos relevantes para cada consulta.

**Chunking (división en fragmentos):**
- Cada sección de las fichas se utiliza como un fragmento independiente, por ejemplo, síntomas, condiciones predisponentes, descripción biológica y cultivos afectados.
- Al comienzo de cada fragmento se incluyen el nombre científico, los nombres comunes y el tipo de información. Esto permite recuperar una ficha aunque la consulta utilice un nombre común, como «roya común».
- Las secciones breves, como tipo, taxonomía, condición y sinonimia, se agrupan en un fragmento denominado «Identidad». La bibliografía se excluye de la búsqueda.
- Los informes de chicharrita se dividen en fragmentos de 1.200 caracteres, con un solapamiento de 200 caracteres entre fragmentos consecutivos.

In [ ]:
NO_INDEXAR = re.compile(r"^bibliograf", re.I)
A_IDENTIDAD = re.compile(r"^(tipo de plaga|taxonom|condici[oó]n$|sinonimia|otros nombres|acr[oó]nimo|nombre)", re.I)
VENTANA, SOLAPE = 1200, 200


def leer_md(p: Path) -> Tuple[dict, str]:
    """Separa el encabezado YAML (metadatos) del cuerpo Markdown."""
    txt = p.read_text(encoding="utf-8")
    meta = {}
    if txt.startswith("---"):
        _, fm, cuerpo = txt.split("---", 2)
        for linea in fm.strip().splitlines():
            if ":" in linea:
                k, v = linea.split(":", 1)
                v = v.strip()
                meta[k.strip()] = ([x.strip().strip('"') for x in v[1:-1].split(",") if x.strip()]
                                   if v.startswith("[") and v.endswith("]") else v.strip('"'))
        return meta, cuerpo
    return meta, txt


def leer_alias(data_dir: Path) -> Dict[str, List[str]]:
    p = data_dir / "plagas_objetivo.csv"
    if not p.exists():
        return {}
    with open(p, encoding="utf-8") as f:
        return {r["slug"]: [a.strip() for a in (r.get("alias_ar") or "").split(";") if a.strip()] for r in csv.DictReader(f)}


def chunks_ficha(meta: dict, cuerpo: str) -> List[dict]:
    nombres = list(dict.fromkeys(meta.get("nombres_vulgares", []) + meta.get("alias_tp", [])))
    encabezado = meta.get("titulo", "") + (f" ({', '.join(nombres)})" if nombres else "")
    etiquetas = [x for x in (meta.get("tipo_plaga"), meta.get("grupo")) if x]
    if etiquetas:
        encabezado += f" [{' · '.join(etiquetas)}]"
    out, identidad = [], []
    for bloque in re.split(r"\n(?=## )", cuerpo):
        if not bloque.startswith("## "):
            continue
        seccion, _, texto = bloque[3:].partition("\n")
        texto = re.sub(r"_Fuente: .*_$", "", texto.strip(), flags=re.M).strip()
        if not texto or NO_INDEXAR.match(seccion):
            continue
        if A_IDENTIDAD.match(seccion):
            identidad.append(f"{seccion}: {' · '.join(l.lstrip('- ') for l in texto.splitlines() if l.strip())}")
            continue
        out.append({"texto": f"{encabezado} — {seccion}\n{texto}", "seccion": seccion})
    out.append({"texto": f"{encabezado} — Identidad\n" + "\n".join(identidad), "seccion": "Identidad"})
    return out


def chunks_informe(meta: dict, cuerpo: str) -> List[dict]:
    out = []
    for pag in re.split(r"<!-- página (\d+) -->", cuerpo)[2::2] or [cuerpo]:
        pag = re.sub(r"[ \t]+", " ", pag).strip()
        for i in range(0, max(len(pag), 1), VENTANA - SOLAPE):
            trozo = pag[i:i + VENTANA].strip()
            if len(trozo) > 80:
                out.append({"texto": f"{meta.get('titulo', '')} ({meta.get('fecha', '')})\n{trozo}", "seccion": "informe"})
    return out


def construir_chunks(data_dir: Path) -> List[dict]:
    alias = leer_alias(data_dir)
    chunks = []
    for p in sorted((data_dir / "corpus").rglob("*.md")):
        meta, cuerpo = leer_md(p)
        slug = meta.get("slug", p.stem)
        meta["alias_tp"] = alias.get(slug, [])
        fuente = p.parent.name
        for c in (chunks_ficha(meta, cuerpo) if fuente == "sinavimo" else chunks_informe(meta, cuerpo)):
            chunks.append({"slug": slug, "fuente": fuente, "titulo": meta.get("titulo", slug),
                           "nombres_vulgares": meta.get("nombres_vulgares", []) + meta["alias_tp"],
                           "cultivos": meta.get("cultivos_tp", []),
                           "fecha": meta.get("fecha") or meta.get("fecha_descarga", ""),
                           "url": meta.get("url", ""), **c})
    return chunks


CHUNKS = construir_chunks(DATA_DIR)
print(len(CHUNKS), "fragmentos")
print(Counter(c["seccion"] for c in CHUNKS).most_common(6))

937 fragmentos
[('informe', 609), ('Identidad', 70), ('Descripción biológica', 69), ('Signos y síntomas / Daños', 69), ('Cultivos / Órgano afectado', 66), ('Condiciones predisponentes', 54)]


### Embeddings y base de datos vectorial (ChromaDB)

Además de BM25, implementamos una búsqueda semántica mediante embeddings, que representan los fragmentos como vectores y permiten recuperar textos similares por su significado. Utilizamos el modelo intfloat/multilingual-e5-base, compatible con español, y almacenamos los vectores en ChromaDB, que realiza la búsqueda por similitud coseno. Elegimos este modelo frente a paraphrase-multilingual-MiniLM-L12-v2 porque, en las pruebas realizadas, recuperó correctamente la ficha esperada con mayor frecuencia.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Modelo multilingüe: convierte cada texto en un vector de 768 números
modelo_embeddings = SentenceTransformer("intfloat/multilingual-e5-base")

# Los modelos e5 se entrenaron con estos prefijos para distinguir preguntas de documentos
PREFIJO_PREGUNTA, PREFIJO_FRAGMENTO = "query: ", "passage: "

# Base vectorial en memoria, con distancia coseno
cliente = chromadb.Client()
if "agro_conocimiento" in [c.name for c in cliente.list_collections()]:
    cliente.delete_collection("agro_conocimiento")      # así la celda se puede volver a correr
coleccion = cliente.create_collection(name="agro_conocimiento", metadata={"hnsw:space": "cosine"})

# Convertimos todos los fragmentos en vectores
textos = [c["texto"] for c in CHUNKS]
vectores = modelo_embeddings.encode([PREFIJO_FRAGMENTO + t for t in textos], normalize_embeddings=True, show_progress_bar=True).tolist()

# Guardamos texto + vector + fuente. El id es la posición del fragmento en CHUNKS,
# así después recuperamos todos sus datos (URL, fecha, cultivos...)
for inicio in range(0, len(CHUNKS), 100):
    fin = min(inicio + 100, len(CHUNKS))
    coleccion.add(
        ids=[str(i) for i in range(inicio, fin)],
        documents=textos[inicio:fin],
        embeddings=vectores[inicio:fin],
        metadatas=[{"fuente": c["fuente"], "slug": c["slug"]} for c in CHUNKS[inicio:fin]],
    )

print("Fragmentos almacenados:", coleccion.count())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Fragmentos almacenados: 937


### Búsqueda con BM25

BM25 asigna una puntuación a cada fragmento según la frecuencia de los términos de la consulta, su relevancia dentro del corpus y la longitud del texto. Para mejorar la recuperación, incorporamos:
- **Normalización del texto:** conversión a minúsculas, eliminación de tildes y palabras vacías, y reducción de palabras a sus raíces.
- **Reconocimiento de nombres comunes:** se priorizan las fichas que coinciden con nombres vulgares mencionados en la consulta.
- **Filtros y actualidad:** se permite filtrar por cultivo y fuente, y priorizar informes recientes cuando la consulta lo requiere.

In [ ]:
STOP = set("""
a al algo algun alguna algunas alguno algunos ante antes como con contra cual cuales cuando de del desde donde dos
el ella ellas ellos en entre era es esa esas ese eso esos esta estan estas este esto estos fue fueron ha hay la las le
les lo los mas me mi mis mucho muy no nos o otra otro para pero poco por porque que quien se sea segun ser si sin
sobre son su sus tambien tan te tengo tiene tienen todo todos tu un una unas uno unos y ya puede pueden hace
veo hoja hojas planta plantas lote cultivo
""".split())
SUFIJOS = ("aciones", "acion", "amientos", "amiento", "mente", "idades", "idad", "osos", "osas", "oso", "osa",
           "ales", "ares", "eres", "iones", "ion", "es", "as", "os", "a", "o", "s")
RECIENTE = re.compile(r"\b(ultim[oa]s?|reciente|recientes|este mes|esta semana|hoy|actual|actualmente|ahora|como viene|como esta)\b")


def normalizar(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", t.lower()) if unicodedata.category(c) != "Mn")


def raiz(p: str) -> str:
    for suf in SUFIJOS:
        if len(p) > len(suf) + 3 and p.endswith(suf):
            return p[: -len(suf)]
    return p


def tokens(t: str) -> List[str]:
    return [raiz(w) for w in re.findall(r"[a-z0-9]+", normalizar(t)) if w not in STOP and len(w) > 1]


class BuscadorBM25:
    def __init__(self, chunks: List[dict], k1: float = 1.4, b: float = 0.75):
        self.chunks, self.k1, self.b = chunks, k1, b
        self.docs = [tokens(c["texto"]) for c in chunks]
        self.N = len(self.docs)
        self.avgdl = sum(map(len, self.docs)) / max(self.N, 1)
        df = Counter(t for d in self.docs for t in set(d))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.tf = [Counter(d) for d in self.docs]
        self.vulgares = {}
        for c in chunks:
            for n in c["nombres_vulgares"] + [c["titulo"]]:
                if len(normalizar(n).strip()) > 4:
                    self.vulgares[normalizar(n).strip()] = c["slug"]
        fechas = sorted({c["fecha"] for c in chunks if c["fuente"] != "sinavimo" and c["fecha"]})
        self.rango_fecha = {f: i / max(len(fechas) - 1, 1) for i, f in enumerate(fechas)}

    def bm25(self, q: List[str]) -> List[float]:
        out = []
        for tf, d in zip(self.tf, self.docs):
            s = 0.0
            for t in q:
                if t in tf:
                    f = tf[t]
                    s += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * len(d) / self.avgdl))
            out.append(s)
        return out

    def buscar(self, consulta: str, k: int = 5, cultivo: Optional[str] = None, fuente: Optional[str] = None,
               mas_reciente: Optional[bool] = None) -> List[dict]:
        if mas_reciente is None:
            mas_reciente = bool(RECIENTE.search(normalizar(consulta)))
        bm = self.bm25(tokens(consulta))
        rank = {i: r for r, i in enumerate(sorted(range(self.N), key=lambda i: bm[i], reverse=True))}
        q = normalizar(consulta)
        mencionados = {s for n, s in self.vulgares.items() if n in q}
        cand = []
        for i, c in enumerate(self.chunks):
            if cultivo and c["cultivos"] and normalizar(cultivo) not in [normalizar(x) for x in c["cultivos"]]:
                continue
            if fuente and c["fuente"] != fuente:
                continue
            if bm[i] == 0 and c["slug"] not in mencionados:
                continue
            s = 1 / (60 + rank[i]) + (0.02 if c["slug"] in mencionados else 0)
            if mas_reciente and c["fuente"] != "sinavimo":
                s += 0.012 * self.rango_fecha.get(c["fecha"], 0)
            cand.append((s, i))
        cand.sort(reverse=True)
        return [{**{k_: self.chunks[i][k_] for k_ in ("slug", "titulo", "seccion", "fuente", "fecha", "url", "texto")},
                 "score": round(s, 4)} for s, i in cand[:k]]


BUSCADOR = BuscadorBM25(CHUNKS)

for r in BUSCADOR.buscar("pústulas color óxido en hojas de maíz", cultivo="maiz", k=3):
    print(f"{r['score']:.4f}  {r['titulo']:<22} [{r['seccion']}]  {r['url']}")

0.0164  Puccinia sorghi        [Signos y síntomas / Daños]  https://www.sinavimo.gob.ar/plaga/puccinia-sorghi
0.0156  Puccinia polysora      [Signos y síntomas / Daños]  https://www.sinavimo.gob.ar/plaga/puccinia-polysora
0.0147  Dalbulus maidis        [Descripción biológica]  https://www.sinavimo.gob.ar/plaga/dalbulus-maidis


### Buscador con embeddings

`BuscadorEmbeddings` tiene el mismo método `buscar()` que `BuscadorBM25`: recibe la consulta y los mismos filtros (cultivo, fuente) y devuelve los fragmentos en el mismo formato. Así las herramientas del agente funcionan igual con cualquiera de los dos buscadores, y para cambiar de uno a otro alcanza con cambiar la variable `BUSCADOR`.

Cuando la pregunta pide lo más reciente ("último informe", "cómo viene"), de los 10 fragmentos más parecidos nos quedamos primero con los de los informes más nuevos.

In [ ]:
class BuscadorEmbeddings:
    """Búsqueda por significado en ChromaDB. Misma interfaz que BuscadorBM25."""

    def __init__(self, chunks, coleccion, modelo):
        self.chunks, self.coleccion, self.modelo = chunks, coleccion, modelo

    def buscar(self, consulta, k=5, cultivo=None, fuente=None, mas_reciente=None):
        if mas_reciente is None:
            mas_reciente = bool(RECIENTE.search(normalizar(consulta)))
        vector = self.modelo.encode([PREFIJO_PREGUNTA + consulta], normalize_embeddings=True).tolist()
        resultados = self.coleccion.query(query_embeddings=vector, n_results=50,
                                          where={"fuente": fuente} if fuente else None)
        encontrados = []
        for id_, distancia in zip(resultados["ids"][0], resultados["distances"][0]):
            c = self.chunks[int(id_)]
            # Filtro por cultivo, igual que en BM25
            if cultivo and c["cultivos"] and normalizar(cultivo) not in [normalizar(x) for x in c["cultivos"]]:
                continue
            encontrados.append({"slug": c["slug"], "titulo": c["titulo"], "seccion": c["seccion"], "fuente": c["fuente"],
                                "fecha": c["fecha"], "url": c["url"], "texto": c["texto"],
                                "score": round(1 - distancia, 4)})      # similitud coseno
        if mas_reciente:
            encontrados = sorted(encontrados[:10], key=lambda r: r["fecha"], reverse=True)
        return encontrados[:k]


BUSCADOR_BM25 = BUSCADOR
BUSCADOR_EMB = BuscadorEmbeddings(CHUNKS, coleccion, modelo_embeddings)

for r in BUSCADOR_EMB.buscar("pústulas color óxido en hojas de maíz", cultivo="maiz", k=3):
    print(f"{r['score']:.4f}  {r['titulo']:<22} [{r['seccion']}]  {r['url']}")

0.8639  Puccinia sorghi        [Signos y síntomas / Daños]  https://www.sinavimo.gob.ar/plaga/puccinia-sorghi
0.8528  Puccinia polysora      [Signos y síntomas / Daños]  https://www.sinavimo.gob.ar/plaga/puccinia-polysora
0.8512  Maize bushy stunt phytoplasma [Signos y síntomas / Daños]  https://www.sinavimo.gob.ar/plaga/maize-bushy-stunt-phytoplasma


### ¿Qué tan bien busca? (evaluación de la recuperación)

Usamos 26 preguntas escritas a mano que describen síntomas sin nombrar la plaga, cada una con la ficha correcta. Medimos:
- **hit@k**: en qué proporción de preguntas aparece la ficha correcta entre los *k* primeros resultados.
- **MRR**: promedio de 1/posición de la ficha correcta (1 = siempre primera).

Con filtro: el buscador sólo mira fichas del cultivo de la pregunta. Por eso le va mejor.
Sin filtro: busca en las 70 fichas. Sirve para ver qué pasa cuando el usuario no dice el cultivo.

In [ ]:
def cargar_jsonl(p: Path) -> List[dict]:
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []

# Cargamos las preguntas de evaluación desde un archivo JSONL. Cada línea del archivo debe ser un objeto JSON que represente una pregunta,
# incluyendo campos como "pregunta", "slugs_esperados" y opcionalmente "cultivo". Si el archivo no existe, se devuelve una lista vacía.
# Ejemplo de línea en el archivo JSONL:
#{"id": "m01", "tipo": "diagnostico_texto", "cultivo": "maiz", "pregunta": "Veo pústulas pulverulentas color canela, como óxido, en ambas caras de las hojas de maíz. ¿Qué puede ser?", "slugs_esperados": ["puccinia-sorghi", "puccinia-polysora"], "comportamiento": "Proponer roya (común o polysora), explicar cómo distinguirlas y citar ficha SINAVIMO."}
PREGUNTAS = cargar_jsonl(DATA_DIR / "evaluacion" / "preguntas_manual.jsonl")

#Compara los resultados con la respuesta correcta(fichas_correctas) y devuelve en qué puesto (1, 2, 3...) aparece la primera ficha correcta, o None si no aparece.
def posicion_ficha_correcta(resultados, fichas_correctas):
    """Devuelve en qué puesto (1, 2, 3...) aparece la primera ficha correcta, o None si no aparece."""
    fichas_vistas = []                      # fichas en el orden en que aparecen, sin repetir
    for fragmento in resultados:
        ficha = fragmento["slug"]
        if ficha not in fichas_vistas:      # una ficha puede tener varios fragmentos: la contamos una vez
            fichas_vistas.append(ficha)

    for puesto, ficha in enumerate(fichas_vistas, start=1):
        if ficha in fichas_correctas:
            return puesto
    return None

#Repetimos posicion_ficha_correcta para  todas las preguntas y calculamos métricas de evaluación: hit@1, hit@3, hit@5 y MRR (Mean Reciprocal Rank).
def evaluar_busqueda(preguntas, buscador, usar_cultivo=True):
    """Mide qué tan bien encuentra el buscador la ficha correcta de cada pregunta."""
    # Sólo sirven las preguntas que tienen una ficha correcta anotada
    preguntas_con_ficha = [p for p in preguntas if p["slugs_esperados"]]

    aciertos_top1 = 0
    aciertos_top3 = 0
    aciertos_top5 = 0
    suma_mrr = 0

    for p in preguntas_con_ficha:
        cultivo = p["cultivo"] if usar_cultivo and p["cultivo"] else None
        resultados = buscador.buscar(p["pregunta"], k=10, cultivo=cultivo)
        puesto = posicion_ficha_correcta(resultados, p["slugs_esperados"])

        if puesto is not None:
            if puesto <= 1:
                aciertos_top1 += 1
            if puesto <= 3:
                aciertos_top3 += 1
            if puesto <= 5:
                aciertos_top5 += 1
            suma_mrr += 1 / puesto           # 1.º vale 1, 2.º vale 0,5, 3.º vale 0,33...

    n = len(preguntas_con_ficha)
    return {
        "n": n,
        "hit@1": round(aciertos_top1 / n, 2),
        "hit@3": round(aciertos_top3 / n, 2),
        "hit@5": round(aciertos_top5 / n, 2),
        "MRR": round(suma_mrr / n, 2),
    }


#Llamamos a evaluar_busqueda para cada buscador (BM25 y Embeddings) y mostramos los resultados con y sin filtro de cultivo.
if PREGUNTAS:
    for nombre, buscador in [("BM25", BUSCADOR_BM25), ("Embeddings", BUSCADOR_EMB)]:
        print(f"{nombre:<11} con filtro de cultivo:", evaluar_busqueda(PREGUNTAS, buscador))
        print(f"{nombre:<11} sin filtro de cultivo:", evaluar_busqueda(PREGUNTAS, buscador, usar_cultivo=False))
else:
    print("No están los sets de evaluación (vienen en la carpeta del Drive).")

BM25        con filtro de cultivo: {'n': 26, 'hit@1': 0.73, 'hit@3': 0.88, 'hit@5': 0.96, 'MRR': 0.82}
BM25        sin filtro de cultivo: {'n': 26, 'hit@1': 0.65, 'hit@3': 0.81, 'hit@5': 0.85, 'MRR': 0.74}
Embeddings  con filtro de cultivo: {'n': 26, 'hit@1': 0.54, 'hit@3': 0.69, 'hit@5': 0.81, 'MRR': 0.65}
Embeddings  sin filtro de cultivo: {'n': 26, 'hit@1': 0.54, 'hit@3': 0.69, 'hit@5': 0.69, 'MRR': 0.63}


## 4. Herramientas (tools)

Además de la búsqueda, el agente puede:
- **Consultar el clima** del lugar con [Open-Meteo](https://open-meteo.com).
- **Definir términos** con el tesauro [AGROVOC de la FAO](https://agrovoc.fao.org).

Es un diccionario muy grande de terminos agricolas organizado relaciones. Devuelve por ejemplo

Término: Roya (en inglés: rusts)
Concepto más amplio: Enfermedad fungosa
Conceptos más específicos: Roya amarilla, Roya del tallo, Roya de las judías
Fuente: AGROVOC (FAO) http://aims.fao.org/aos/agrovoc/c_6710

- **Evaluar riesgo** (*skill*): junta las "condiciones predisponentes" de la ficha con el pronóstico del lugar.

Cada tool devuelve **texto corto con la URL de la fuente**, para que el modelo pueda citar.

In [ ]:
SESION = requests.Session()
SESION.headers["User-Agent"] = "TP-IA-agente-agro/1.0 (uso academico)"   # nos identificamos ante las APIs

CACHE = {}   # guarda las respuestas ya pedidas, para no repetir la misma consulta a internet

#Hace una consulta a una pagina y guarda en el cache, reintentando si falla. Devuelve el JSON de la respuesta.
def http_get(url, params=None, intentos=4):
    """Hace un pedido GET y devuelve el JSON. Guarda el resultado en CACHE y reintenta si la API falla."""
    clave = url + str(sorted((params or {}).items()))
    if clave in CACHE:
        return CACHE[clave]

    for intento in range(1, intentos + 1):
        try:
            respuesta = SESION.get(url, params=params, timeout=30)
            respuesta.raise_for_status()            # error si la API respondió con un código de error (404, 429, 500...)
            CACHE[clave] = respuesta.json()
            return CACHE[clave]
        except requests.RequestException:
            if intento == intentos:                 # último intento: no insistimos más
                raise
            time.sleep(5 * intento)                 # esperamos 5, 10, 15 s antes de volver a probar

# Le tiro un nombre de localidad y me da las coordenadas
def geocodificar(lugar):
    """Nombre de una localidad argentina -> sus coordenadas (latitud y longitud)."""
    datos = http_get("https://geocoding-api.open-meteo.com/v1/search",
                     {"name": lugar, "count": 3, "language": "es", "countryCode": "AR"})
    resultados = datos.get("results") or []
    if not resultados:
        return {"error": f"No encontré la localidad '{lugar}' en Argentina."}

    primero = resultados[0]                         # el resultado más probable
    return {"nombre": primero["name"], "provincia": primero.get("admin1"),
            "lat": primero["latitude"], "lon": primero["longitude"]}

# Las variables climáticas que pedimos
VARIABLES_CLIMA = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
                   "relative_humidity_2m_mean", "leaf_wetness_probability_mean"]

#Consulta el pronóstico diario de Open-Meteo para una latitud y longitud dadas, devolviendo una lista de diccionarios con los datos diarios.
def pronostico(lat, lon, dias=7):
    """Pronóstico diario de Open-Meteo: una fila (diccionario) por día."""
    dias = max(1, min(dias, 16))                    # Open-Meteo pronostica entre 1 y 16 días
    datos = http_get("https://api.open-meteo.com/v1/forecast",
                     {"latitude": lat, "longitude": lon, "daily": ",".join(VARIABLES_CLIMA),
                      "forecast_days": dias, "timezone": "America/Argentina/Cordoba"})
    diario = datos["daily"]                         # una lista por variable: {"time": [...], "precipitation_sum": [...]}

    # Lo damos vuelta: una fila por día con todas sus variables
    filas = []
    for i, fecha in enumerate(diario["time"]):
        fila = {"fecha": fecha}
        for variable in VARIABLES_CLIMA:
            fila[variable] = diario[variable][i]
        filas.append(fila)
    return filas


AGROVOC = "https://agrovoc.fao.org/browse/rest/v1"


def como_lista(valor):
    """AGROVOC a veces devuelve un solo elemento y a veces una lista: lo dejamos siempre como lista."""
    if valor is None:
        return []
    if isinstance(valor, dict):
        return [valor]
    return valor


def nombres_en(concepto, idioma):
    """Nombres de un concepto de AGROVOC en un idioma ('es' o 'en')."""
    return [n["value"] for n in como_lista(concepto.get("prefLabel")) if n.get("lang") == idioma]


def glosario_agrovoc(termino):
    """Busca un término en AGROVOC (FAO): nombre en español e inglés, concepto más amplio y más específicos."""
    # 1. Buscar el término y quedarnos con el primer resultado
    busqueda = http_get(f"{AGROVOC}/search/", {"query": termino + "*", "lang": "es", "maxhits": 5})
    resultados = busqueda.get("results", [])
    if not resultados:
        return {"error": f"'{termino}' no está en AGROVOC."}
    uri = resultados[0]["uri"]                      # identificador del concepto, ej. .../c_6710

    # 2. Pedir los datos del concepto. Vienen como una lista de conceptos (él y sus relacionados)
    datos = http_get(f"{AGROVOC}/agrovoc/data", {"uri": uri, "format": "application/ld+json"})
    conceptos = {c.get("uri"): c for c in datos.get("graph", [])}
    concepto = conceptos.get(uri, {})

    # 3. Pasar de los identificadores de los relacionados a sus nombres en español
    def nombres_relacionados(relacion):            # relacion: "broader" (más amplio) o "narrower" (más específico)
        nombres = []
        for rel in como_lista(concepto.get(relacion)):
            es = nombres_en(conceptos.get(rel["uri"], {}), "es")
            nombres.append(es[0] if es else rel["uri"])
        return nombres

    return {"uri": uri,
            "es": nombres_en(concepto, "es"),
            "en": nombres_en(concepto, "en"),
            "mas_amplio": nombres_relacionados("broader"),
            "mas_especifico": nombres_relacionados("narrower")[:8]}


print(geocodificar("Río Cuarto"))
print(glosario_agrovoc("roya"))

{'nombre': 'Ciudad de Río Cuarto', 'provincia': 'Provincia de Córdoba', 'lat': -33.13044, 'lon': -64.35272}
{'uri': 'http://aims.fao.org/aos/agrovoc/c_6710', 'es': ['Roya'], 'en': ['rusts'], 'mas_amplio': ['Enfermedad fungosa'], 'mas_especifico': ['Roya amarilla', 'Roya del tallo', 'Roya de las judías']}


In [ ]:
MAX_CHARS_FRAGMENTO = 700        # cada fragmento se recorta a este largo antes de dárselo al modelo
CULTIVOS = {"maiz", "soja", "trigo", "mani"}


def _cultivo(cultivo):
    """Normaliza el cultivo que escribe el modelo ("Maíz" -> "maiz"). Si no es uno de los 4, devuelve None (sin filtro)."""
    if not cultivo:
        return None
    cultivo = normalizar(cultivo).strip()
    if cultivo in CULTIVOS:
        return cultivo
    return None


def recortar(texto, largo=MAX_CHARS_FRAGMENTO):
    """Corta el texto en `largo` caracteres sin partir la última palabra, y marca el corte con […]."""
    if len(texto) <= largo:
        return texto
    cortado = texto[:largo]
    sin_ultima_palabra = cortado.rsplit(" ", 1)[0]
    return sin_ultima_palabra + " […]"


def formatear(resultados):
    """Convierte los fragmentos encontrados en el texto que lee el modelo: número, título, URL y contenido."""
    if not resultados:
        return "Sin resultados en las fuentes oficiales disponibles."

    bloques = []
    for numero, r in enumerate(resultados, start=1):
        encabezado = f"[{numero}] {r['titulo']} — {r['seccion']}"
        if r["fuente"] != "sinavimo":                       # los informes llevan fecha; las fichas no
            encabezado += f" · fecha {r['fecha']}"
        bloques.append(f"{encabezado}\nURL: {r['url']}\n{recortar(r['texto'])}")
    return "\n\n".join(bloques)


# ---------- Herramientas de búsqueda (RAG) ----------

def consultar_fichas(consulta, cultivo=None, k=4):
    """Busca en las fichas de SINAVIMO."""
    resultados = BUSCADOR.buscar(consulta, k=k, cultivo=_cultivo(cultivo), fuente="sinavimo")
    return formatear(resultados)


def situacion_chicharrita(consulta="situación actual", k=3):
    """Busca en los informes de chicharrita, priorizando los más recientes."""
    resultados = BUSCADOR.buscar(consulta, k=k, fuente="chicharrita", mas_reciente=True)
    return formatear(resultados)


# ---------- Herramienta de clima ----------

def clima_localidad(localidad, dias=7):
    """Pronóstico de una localidad: primero un resumen y después el detalle día por día."""
    try:
        lugar = geocodificar(localidad)
        if "error" in lugar:
            return lugar["error"]
        dias_pronostico = pronostico(lugar["lat"], lugar["lon"], dias)
    except requests.RequestException as e:
        return f"No pude obtener el pronóstico en este momento ({type(e).__name__}). Respondé sin datos de clima y aclaralo."

    # Resumen de todo el período (algunos días pueden venir sin dato: None)
    lluvia_total = 0
    humedades = []
    mojados = []
    for dia in dias_pronostico:
        lluvia_total += dia["precipitation_sum"] or 0
        if dia["relative_humidity_2m_mean"] is not None:
            humedades.append(dia["relative_humidity_2m_mean"])
        if dia["leaf_wetness_probability_mean"] is not None:
            mojados.append(dia["leaf_wetness_probability_mean"])

    humedad_media = sum(humedades) / len(humedades) if humedades else 0
    dias_humedos = len([h for h in humedades if h >= 80])        # días con humedad relativa de 80% o más
    dias_hoja_mojada = len([m for m in mojados if m >= 60])      # días con probabilidad de mojado de hoja de 60% o más

    # Detalle: una línea por día
    lineas = []
    for dia in dias_pronostico:
        lineas.append(f"{dia['fecha']}: Tmax {dia['temperature_2m_max']}°C / Tmin {dia['temperature_2m_min']}°C, "
                      f"lluvia {dia['precipitation_sum']} mm, HR {dia['relative_humidity_2m_mean']}%, "
                      f"prob. mojado de hoja {dia['leaf_wetness_probability_mean']}%")
    detalle = "\n".join(lineas)

    return (f"Localidad: {lugar['nombre']} ({lugar['provincia']})\n"
            f"Próximos {len(dias_pronostico)} días: lluvia total {lluvia_total:.1f} mm, HR media {humedad_media:.0f}%, "
            f"días con HR≥80%: {dias_humedos}, días con mojado de hoja ≥60%: {dias_hoja_mojada}\n"
            f"{detalle}\n"
            f"Fuente: Open-Meteo (https://open-meteo.com), pronóstico.")


# ---------- Herramienta de glosario ----------

def definir_termino(termino):
    """Nombre en español e inglés de un término y sus conceptos relacionados, según AGROVOC (FAO)."""
    try:
        g = glosario_agrovoc(termino)
    except requests.RequestException as e:
        return f"No pude consultar AGROVOC en este momento ({type(e).__name__})."
    if "error" in g:
        return g["error"]

    mas_amplio = ", ".join(g["mas_amplio"]) or "-"
    mas_especifico = ", ".join(g["mas_especifico"]) or "-"
    return (f"Término: {', '.join(g['es'])} (en inglés: {', '.join(g['en'])})\n"
            f"Concepto más amplio: {mas_amplio}\n"
            f"Conceptos más específicos: {mas_especifico}\n"
            f"Fuente: AGROVOC (FAO) {g['uri']}")


# ---------- Skill: riesgo = ficha + clima ----------

def evaluar_riesgo(plaga, localidad, cultivo=None, dias=7):
    """SKILL: pone lado a lado las condiciones que favorecen la plaga (ficha) y el pronóstico, para que el LLM compare."""
    # 1. Buscar en las fichas las condiciones predisponentes de la plaga
    resultados = BUSCADOR.buscar(f"{plaga} condiciones predisponentes temperatura humedad",
                                 k=3, cultivo=_cultivo(cultivo), fuente="sinavimo")
    condiciones = [r for r in resultados if r["seccion"].lower().startswith("condiciones")]
    if not condiciones:                    # si ninguna sección es "Condiciones...", usamos el mejor resultado
        condiciones = resultados[:1]

    # 2. Pronóstico del lugar
    clima = clima_localidad(localidad, dias)

    # 3. Armar el texto para el modelo
    return ("=== CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===\n"
            + formatear(condiciones[:2])
            + "\n\n=== PRONÓSTICO DEL LUGAR ===\n"
            + clima
            + "\n\nCompará los rangos de la ficha con el pronóstico y decí si el riesgo es bajo, medio o alto, "
              "con números. Es una estimación orientativa.")


def fecha_actual(_=""):
    """Fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")


print(consultar_fichas("manchas circulares con borde rojizo", cultivo="soja", k=1))
print()
print(evaluar_riesgo("mancha amarilla del trigo", "Pergamino", cultivo="trigo", dias=3)[:900])

[1] Cercospora sojina — Signos y síntomas / Daños
URL: https://www.sinavimo.gob.ar/plaga/cercospora-sojina
Cercospora sojina (Mancha ojo de rana) [Hongos y/ó Pseudohongos · hongo] — Signos y síntomas / Daños
Esta enfermedad afecta fundamentalmente al follaje, aunque también puede infectar tallos, vainas y semillas. Las hojas presentan lesiones circulares a angulares, que varían en color, desde rojizo en las lesiones más pequeñas (1 a 2 mm.) a marrón oscuro o claro en las lesiones más grandes (3 a 5 mm.), las que están rodeadas de un borde marrón rojizo. La ausencia de un halo amarillento alrededor de las manchas es una característica distintiva de la enfermedad. En la cara inferior de las hojas, sobre el centro de las lesiones, se observan puntos grises a negros. Cuando se presenta un gran […]

=== CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===
[1] Pyrenophora tritici-repentis — Condiciones predisponentes
URL: https://www.sinavimo.gob.ar/plaga/pyrenophora-tritici-repentis
Pyren

### Guardrails (barreras de seguridad): el agente no receta
Para evitar recomendaciones de productos fitosanitarios y dosis de aplicación, incorporamos barreras de seguridad en el agente. Además de las instrucciones incluidas en el prompt, implementamos dos controles adicionales:
- Control previo: si la consulta solicita un producto o una dosis, se agrega una advertencia antes de enviar el mensaje al modelo.
- Control posterior: si la respuesta generada contiene una dosis de aplicación (por ejemplo, l/ha, cc/ha o g/ha), se reemplaza por una derivación a un profesional.
De esta manera, el sistema no depende únicamente de que el modelo siga las instrucciones, sino que incorpora verificaciones adicionales para reducir el riesgo de recomendaciones inadecuadas.

In [ ]:
# ---------- Detectar si la PREGUNTA pide productos o dosis (antes del LLM) ----------

# Frases típicas de un pedido de dosis o producto. [aá] acepta la palabra con o sin tilde; (le )? hace opcional el "le".
FRASES_PEDIDO_DOSIS = [
    r"dosis",
    r"cu[aá]nto (le )?(aplico|pongo|echo|tiro)",                                       # "cuánto le aplico"
    r"qu[eé] (producto|insecticida|fungicida|herbicida|le aplico|le pongo|le tiro)",   # "qué insecticida"
    r"con qu[eé] (lo )?(curo|controlo|fumigo|mato)",                                   # "con qué lo controlo"
    r"cu[aá]ntos? (litros|cc|gramos)",                                                 # "cuántos litros"
]
# \b = borde de palabra (que no sea parte de otra palabra); re.I = sin importar mayúsculas
PEDIDO_DOSIS = re.compile(r"\b(" + "|".join(FRASES_PEDIDO_DOSIS) + r")\b", re.I)


def pide_dosis(pregunta):
    """True si la pregunta pide productos o dosis."""
    return PEDIDO_DOSIS.search(pregunta) is not None


# ---------- Detectar si la RESPUESTA trae una dosis (después del LLM) ----------

# Una dosis se escribe: número + unidad + "por" + hectárea. Ej: "0,5 l/ha", "500 cc por hectárea"
NUMERO = r"\d+(?:[.,]\d+)?"                                     # 500, 0,5 o 1.5
UNIDAD = r"(?:l|lt|litros?|cc|ml|g|gr|gramos?|kg|cm3)"
POR = r"(?:/|por|x)"
HECTAREA = r"(?:ha|hect[aá]reas?)"
DOSIS_EN_RESPUESTA = re.compile(rf"\b{NUMERO}\s*{UNIDAD}\s*{POR}\s*{HECTAREA}\b", re.I)   # \s* = espacios opcionales


# ---------- Textos que usan los controles ----------

URL_CASAFE = "https://www.casafe.org/publicaciones/guia-de-productos-fitosanitarios/"

# Reemplaza la respuesta completa si el modelo igual dio una dosis
DERIVACION = ("No puedo indicar productos, principios activos ni dosis: una aplicación de fitosanitarios requiere la receta "
              "de un Ingeniero Agrónomo matriculado y respetar la etiqueta del producto registrado en SENASA (Guía de Productos "
              f"Fitosanitarios de CASAFE: {URL_CASAFE}). Sí puedo ayudarte "
              "con el manejo integrado: monitoreo, umbrales, condiciones que favorecen la plaga y prácticas culturales.")

# Se agrega a la pregunta antes de mandarla al modelo, cuando pide_dosis() da True
NOTA_DOSIS = ("[Nota del sistema: el usuario pide productos o dosis. NO los indiques. Explicá manejo integrado usando las fichas "
              "y derivá a un Ingeniero Agrónomo, la etiqueta registrada y la receta fitosanitaria.]")


def revisar_respuesta(respuesta):
    """Si la respuesta del modelo trae una dosis, la cambia por la derivación al profesional."""
    if DOSIS_EN_RESPUESTA.search(respuesta):
        return DERIVACION
    return respuesta


print(pide_dosis("¿Qué dosis de insecticida le pongo?"), pide_dosis("¿Qué favorece la roya?"))
print(revisar_respuesta("Aplicá 0,5 l/ha")[:60], "…")
print(revisar_respuesta("Aplicá 500 cc por hectárea")[:60], "…")

True False
No puedo indicar productos, principios activos ni dosis: una …
No puedo indicar productos, principios activos ni dosis: una …


## 5. Conexión con el modelo (local, en la GPU de Colab)

Utilizamos Qwen/Qwen3-4B-Instruct-2507, ejecutado localmente en la GPU de Google Colab mediante transformers, para evitar el uso de créditos de API. La ejecución requiere una GPU y la descarga inicial del modelo.
- generar() aplica el formato de conversación de Qwen y permite generar respuestas con herramientas.
- call_llm() corresponde a la versión 1 del agente, con un máximo de 256 tokens y temperatura 0,1. Esta configuración se mantiene para comparar sus limitaciones con las mejoras incorporadas en la versión 2 (sección 7).

In [ ]:
# Necesita GPU: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

if not torch.cuda.is_available():
    print("Atención: no hay GPU. Activá una T4 en 'Entorno de ejecución → Cambiar tipo de entorno' ")

#Seguimos el ejemplo del repositorio de la catedra
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", dtype=torch.bfloat16)
print(f"Modelo {MODEL_ID} cargado en {model.device}")

def generar(messages, temperature=0.0, max_new_tokens=768, stop=None):
    """Recibe la conversación (lista de mensajes) y devuelve la respuesta del modelo como texto.
    temperature = 0  -> siempre elige la palabra más probable (misma pregunta, misma respuesta). A medida que aumentamos la temperatura, el modelo se vuelve más creativo y menos predecible.
    stop             -> textos en los que cortar la respuesta (la versión 2 corta en "Observation:")."""
    # Pasar la conversación al formato de chat que espera Qwen (marca qué dijo el sistema, el usuario y el asistente)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Texto pasado a tokens (números). add_special_tokens=False porque el chat template ya agregó los tokens especiales
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)

    # Opciones de generación
    opciones = {"max_new_tokens": max_new_tokens, "pad_token_id": tokenizer.eos_token_id}
    if temperature > 0:
        opciones.update(do_sample=True, temperature=temperature, top_p=0.8)     # con algo de azar
    else:
        opciones.update(do_sample=False, temperature=None, top_p=None, top_k=None)   # greedy, sin azar
    if stop:
        opciones.update(stop_strings=stop, tokenizer=tokenizer)

    # Generar. no_grad: sólo usamos el modelo, no lo entrenamos, así gasta menos memoria
    with torch.no_grad():
        salida = model.generate(**inputs, **opciones)

    # La salida trae la pregunta + la respuesta: nos quedamos sólo con los tokens nuevos y los pasamos a texto
    largo_prompt = inputs["input_ids"].shape[1]
    tokens_nuevos = salida[0][largo_prompt:]
    texto = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)

    if len(tokens_nuevos) >= max_new_tokens:
        print(f"[aviso] la respuesta llegó al límite de {max_new_tokens} tokens y puede estar cortada")

    # Si generó el texto de corte (ej. "Observation:"), sacamos eso y lo que sigue
    for s in stop or []:
        texto = texto.split(s)[0]
    return texto.strip()


def call_llm(messages, temperature=0.1):
    """LLM de la versión 1 del agente: hasta 256 tokens y temperatura 0,1 (la configuración de la primera corrida).
    La versión 2 (sección 7) usa call_llm_v2."""
    return generar(messages, temperature=temperature, max_new_tokens=256)


print(call_llm([{"role": "user", "content": "Decí en una oración qué es una plaga agrícola."}]))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Modelo Qwen/Qwen3-4B-Instruct-2507 cargado en cuda:0
Una plaga agrícola es un conjunto de organismos, como insectos, plagas o enfermedades, que dañan los cultivos y afectan negativamente la producción agrícola.


## 6. Agente ReAct manual — versión 1

En esta versión implementamos un ciclo ReAct en el que el modelo propone una acción mediante texto, pero no ejecuta directamente las herramientas. Utiliza un formato estructurado:
```text
Thought: razonamiento breve
Action: nombre_tool | input | parametro=valor
```
Python identifica la acción mediante una expresión regular, ejecuta la herramienta correspondiente y devuelve su resultado como `Observation`. El ciclo sigue hasta que el modelo responde `FINAL: ...`.

In [ ]:
@dataclass
class Tool:
    """Una herramienta del agente: nombre, descripción (la lee el modelo en el prompt) y la función de Python que se ejecuta."""
    name: str
    description: str
    func: Callable[[str], str]


def parse_tool_input(texto):
    """Separa lo que escribió el modelo después de 'Action: herramienta |'.
    'roya común | cultivo=maiz | k=3'  ->  ('roya común', {'cultivo': 'maiz', 'k': '3'})"""
    partes = [p.strip() for p in texto.split("|")]
    consulta = partes[0]                    # lo primero es siempre la consulta principal
    parametros = {}
    for parte in partes[1:]:                # lo demás son parámetros opcionales del tipo clave=valor
        if "=" in parte:
            clave, valor = parte.split("=", 1)
            parametros[clave.strip().lower()] = valor.strip()
    return consulta, parametros


def _int(valor, defecto):
    """Convierte a número entero; si el modelo escribió algo raro (o nada), usa el valor por defecto."""
    try:
        return int(valor)
    except (TypeError, ValueError):
        return defecto


# Cada función t_... recibe el texto que escribió el modelo, lo separa y llama a la herramienta real (sección 4)

def t_buscar_fichas(texto):
    consulta, p = parse_tool_input(texto)
    return consultar_fichas(consulta, p.get("cultivo"), _int(p.get("k"), 4))


def t_chicharrita(texto):
    consulta, p = parse_tool_input(texto)
    return situacion_chicharrita(consulta or "situación actual", _int(p.get("k"), 3))


def t_clima(texto):
    localidad, p = parse_tool_input(texto)
    return clima_localidad(localidad, _int(p.get("dias"), 7))


def t_riesgo(texto):
    plaga, p = parse_tool_input(texto)
    if not p.get("localidad"):              # sin localidad no hay pronóstico: le explicamos al modelo cómo pedirla
        return "Error: falta localidad. Formato: Action: evaluar_riesgo | plaga | localidad=Río Cuarto | cultivo=maiz"
    return evaluar_riesgo(plaga, p["localidad"], p.get("cultivo"), _int(p.get("dias"), 7))


def t_glosario(texto):
    termino, _ = parse_tool_input(texto)
    return definir_termino(termino)


# Catálogo de herramientas. La descripción se copia al prompt: el modelo sólo sabe usar lo que está escrito ahí
TOOLS = {
    "buscar_fichas": Tool(
        "buscar_fichas",
        "Busca en las fichas oficiales de plagas, enfermedades y malezas (SINAVIMO-SENASA): síntomas, biología, condiciones "
        "predisponentes, cultivos afectados y presencia en Argentina. Usar SIEMPRE antes de dar información sobre una plaga.\n"
        "Formato: Action: buscar_fichas | consulta | cultivo=maiz|soja|trigo|mani (opcional)\n"
        "Ejemplo: Action: buscar_fichas | manchas circulares con borde rojizo en hojas | cultivo=soja",
        t_buscar_fichas),
    "situacion_chicharrita": Tool(
        "situacion_chicharrita",
        "Informes más recientes de la Red Nacional de Monitoreo de la chicharrita del maíz (capturas por región, % de Spiroplasma).\n"
        "Formato: Action: situacion_chicharrita | región o tema\n"
        "Ejemplo: Action: situacion_chicharrita | capturas región Centro Norte",
        t_chicharrita),
    "clima_lote": Tool(
        "clima_lote",
        "Pronóstico de los próximos días (temperatura, lluvia, humedad, mojado de hoja) para una localidad argentina.\n"
        "Formato: Action: clima_lote | localidad | dias=7\n"
        "Ejemplo: Action: clima_lote | Marcos Juárez | dias=5",
        t_clima),
    "evaluar_riesgo": Tool(
        "evaluar_riesgo",
        "Skill: condiciones que favorecen una plaga (ficha oficial) + pronóstico de una localidad, para estimar el riesgo.\n"
        "Formato: Action: evaluar_riesgo | plaga | localidad=... | cultivo=... (opcional)\n"
        "Ejemplo: Action: evaluar_riesgo | mancha amarilla del trigo | localidad=Pergamino | cultivo=trigo",
        t_riesgo),
    "glosario": Tool(
        "glosario",
        "Define o traduce un término agronómico (AGROVOC, FAO).\n"
        "Formato: Action: glosario | término\n"
        "Ejemplo: Action: glosario | umbral económico",
        t_glosario),
    "fecha_actual": Tool(
        "fecha_actual",
        "Fecha y hora actual. Formato: Action: fecha_actual |",
        fecha_actual),
}

# Texto con todas las herramientas, que se inserta en el system prompt
TOOLS_DESC = "\n\n".join(f"- {t.name}: {t.description}" for t in TOOLS.values())
print(TOOLS_DESC)

- buscar_fichas: Busca en las fichas oficiales de plagas, enfermedades y malezas (SINAVIMO-SENASA): síntomas, biología, condiciones predisponentes, cultivos afectados y presencia en Argentina. Usar SIEMPRE antes de dar información sobre una plaga.
Formato: Action: buscar_fichas | consulta | cultivo=maiz|soja|trigo|mani (opcional)
Ejemplo: Action: buscar_fichas | manchas circulares con borde rojizo en hojas | cultivo=soja

- situacion_chicharrita: Informes más recientes de la Red Nacional de Monitoreo de la chicharrita del maíz (capturas por región, % de Spiroplasma).
Formato: Action: situacion_chicharrita | región o tema
Ejemplo: Action: situacion_chicharrita | capturas región Centro Norte

- clima_lote: Pronóstico de los próximos días (temperatura, lluvia, humedad, mojado de hoja) para una localidad argentina.
Formato: Action: clima_lote | localidad | dias=7
Ejemplo: Action: clima_lote | Marcos Juárez | dias=5

- evaluar_riesgo: Skill: condiciones que favorecen una plaga (ficha oficia

In [ ]:
SYSTEM_PROMPT = f"""
Sos AgroAgente, un agente ReAct que responde consultas fitosanitarias sobre maíz, soja, trigo y maní en Argentina.

==================================================
HERRAMIENTAS DISPONIBLES
==================================================

{TOOLS_DESC}

==================================================
FORMATO OBLIGATORIO
==================================================

Si necesitás una herramienta respondé EXACTAMENTE así y nada más:

Thought: razonamiento breve
Action: nombre_tool | input

Cuando tengas suficiente información respondé EXACTAMENTE:

FINAL: respuesta final para el usuario

==================================================
REGLAS
==================================================

1. SOLO podés usar: {", ".join(TOOLS)}. NO inventes herramientas.
2. NUNCA escribas Observation: la agrega Python.
3. Una sola Action por respuesta. La Action SIEMPRE lleva "|".
4. Antes de afirmar síntomas, condiciones o datos de una plaga, usá buscar_fichas. No respondas de memoria.
5. Basá la respuesta FINAL sólo en las Observations. Si no alcanzan, decí que no encontraste la información.
6. En la respuesta FINAL citá las fuentes con su URL, así: (Fuente: URL).
7. NUNCA indiques productos, principios activos ni dosis. Recomendá manejo integrado (monitoreo, umbrales,
   rotación, prácticas culturales) y derivá a un Ingeniero Agrónomo y a la etiqueta del producto registrado.
8. Si la consulta no es agronómica (precios, mercados, otros temas), respondé FINAL diciendo que está fuera de tu alcance.
9. Si falta un dato clave (por ejemplo el cultivo), podés preguntarlo en la respuesta FINAL.
10. Para diagnósticos por síntomas, aclarales que es orientativo y que conviene confirmarlo a campo.
11. Respondé en español rioplatense, claro y breve.

==================================================
EJEMPLO
==================================================

Thought: Necesito los síntomas de la roya común en las fichas.
Action: buscar_fichas | pústulas color óxido en hojas | cultivo=maiz

Observation: [1] Puccinia sorghi — Signos y síntomas / Daños
URL: https://www.sinavimo.gob.ar/plaga/puccinia-sorghi
...

Thought: Ya tengo la información de la ficha.
FINAL: Lo más probable es roya común del maíz (Puccinia sorghi)... (Fuente: https://www.sinavimo.gob.ar/plaga/puccinia-sorghi)
""".strip()


# Busca una línea que empiece con "Action:" y separa el nombre de la herramienta de lo que va después de la "|".
ACTION_RE = re.compile(r"^\s*Action:\s*([\w\-]+)\s*\|\s*(.*?)\s*$", re.MULTILINE)
# Busca "FINAL:" (lo que pide el prompt) o "Final Answer:" (lo que buscaba el parser de la cátedra).
FINAL_RE = re.compile(r"(?:^|\n)\s*(?:FINAL|Final Answer)\s*:\s*(.*)", re.DOTALL | re.IGNORECASE)


def parse_action(salida_llm):
    """Si el modelo pidió una herramienta, devuelve (nombre, entrada). Si no, devuelve None."""
    encontrado = ACTION_RE.search(salida_llm)
    if encontrado is None:
        return None
    nombre = encontrado.group(1).strip()
    entrada = encontrado.group(2).strip()
    return nombre, entrada

def parse_final_answer(salida_llm):
    """Si el modelo dio la respuesta final, devuelve su texto. Si no, devuelve None."""
    encontrado = FINAL_RE.search(salida_llm)
    if encontrado is None:
        return None
    return encontrado.group(1).strip()


print(parse_action("Thought: busco\nAction: buscar_fichas | roya | cultivo=maiz"))
print(parse_final_answer("Thought: listo\nFINAL: Es roya común."))

('buscar_fichas', 'roya | cultivo=maiz')
Es roya común.


### Bucle ReAct

El modelo propone una acción, Python ejecuta la herramienta correspondiente y devuelve el resultado como `Observation`. El ciclo se repite hasta que el modelo genera una respuesta final `FINAL:` o se alcanza el límite de pasos. La implementación incorpora tres mejoras:
- Memoria: conserva las últimas cuatro preguntas y respuestas.
- Robustez: si se solicita una herramienta inexistente o se produce un error durante su ejecución, se devuelve el mensaje como `Observation`para que el modelo pueda corregirse.
- Guardrails: incorpora controles de seguridad antes y después de generar la respuesta.

In [ ]:
class AgroAgenteReAct:
    """Agente ReAct, versión 1. Mismo bucle que run_agent de la cátedra, más memoria, manejo de errores y guardrails."""

    def __init__(self, llm=None, max_steps=6, max_obs=3500):
        self.llm = llm or call_llm      # función que recibe la conversación y devuelve lo que escribe el modelo
        self.max_steps = max_steps      # máximo de vueltas del bucle por pregunta
        self.max_obs = max_obs          # largo máximo de cada Observation (para no llenar el contexto del modelo)
        self.historial = []             # memoria: preguntas y respuestas anteriores

    def reiniciar(self):
        """Borra la memoria (empezar una conversación nueva)."""
        self.historial = []

    def ejecutar_herramienta(self, nombre, entrada):
        """Corre la herramienta pedida y devuelve su resultado. Si algo sale mal, devuelve el error como texto
        para que el modelo lo lea y pueda corregirse."""
        if nombre not in TOOLS:
            return f"Error: la herramienta '{nombre}' no existe. Herramientas disponibles: {list(TOOLS)}"
        try:
            resultado = TOOLS[nombre].func(entrada)
        except Exception as e:
            resultado = f"Error al ejecutar {nombre}: {type(e).__name__}: {e}"
        return resultado[: self.max_obs]

    def recordar(self, pregunta, respuesta):
        """Guarda la pregunta y la respuesta en la memoria. Se quedan sólo los últimos 8 mensajes (4 preguntas)."""
        self.historial.append({"role": "user", "content": pregunta})
        self.historial.append({"role": "assistant", "content": f"FINAL: {respuesta}"})
        self.historial = self.historial[-8:]

    def preguntar(self, pregunta):
        """Responde una pregunta. Devuelve (respuesta, traza)."""
        # Guardrail de entrada: si pide dosis, le agregamos una nota al mensaje
        contenido = pregunta
        if pide_dosis(pregunta):
            contenido += f"\n\n{NOTA_DOSIS}"

        # Conversación: instrucciones + memoria + pregunta nueva
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        messages += self.historial
        messages.append({"role": "user", "content": contenido})

        traza = []                      # lo que pasó en cada paso, para poder mostrarlo después
        respuesta = None

        for paso in range(1, self.max_steps + 1):
            # 1. El modelo escribe
            salida = self.llm(messages)
            messages.append({"role": "assistant", "content": salida})

            # 2. ¿Pidió una herramienta o dio la respuesta final?
            accion = parse_action(salida)
            final = parse_final_answer(salida)
            # Si escribió las dos cosas, gana la que aparece primero en el texto
            # (ojo: busca "FINAL" literal, así que no ve "Final Answer"; la versión 2 corrige esto)
            final_va_primero = bool(final) and salida.find("FINAL") < salida.find("Action")

            if accion and not final_va_primero:
                # 3a. Pidió una herramienta: Python la ejecuta y le devuelve el resultado como Observation
                nombre, entrada = accion
                observacion = self.ejecutar_herramienta(nombre, entrada)
                traza.append({"step": paso, "llm_output": salida, "action": nombre, "input": entrada,
                              "observation": observacion})
                messages.append({"role": "user",
                                 "content": f"Observation: {observacion}\n\nContinuá. Si ya podés responder, usá FINAL:"})
                continue                # otra vuelta: el modelo lee la Observation

            # 3b. Respondió (o no siguió el formato): usamos lo que escribió como respuesta
            if final:
                respuesta = final
            else:
                respuesta = salida.strip()
            respuesta = revisar_respuesta(respuesta)        # guardrail de salida: si trae una dosis, la reemplaza
            traza.append({"step": paso, "llm_output": salida, "action": None, "observation": None})
            break

        if respuesta is None:           # se terminaron los pasos sin respuesta final
            respuesta = "No llegué a una respuesta final dentro del máximo de pasos. ¿Podés reformular la consulta?"

        self.recordar(pregunta, respuesta)
        return respuesta, traza


def print_trace(traza):
    """Muestra paso a paso lo que escribió el modelo y qué herramientas se ejecutaron."""
    for fila in traza:
        print(f"\n--- Paso {fila['step']} ---\n{fila['llm_output']}")
        if fila.get("action"):
            print(f"\n[Tool ejecutada] {fila['action']} | {fila['input']}\nObservation: {fila['observation'][:600]}…")


agente = AgroAgenteReAct()

### Prueba del agente versión 1

Evaluamos el agente con cuatro preguntas representativas de los tipos de consultas que debe resolver. Para cada una, definimos previamente la herramienta que debía utilizar, la fuente que debía consultar y el contenido esperado de la respuesta.
Las trazas permiten seguir los pasos del modelo y verificar qué herramientas se ejecutaron. Cada `[Tool ejecutada]`indica que se realizó una consulta real; si el agente responde directamente con  `FINAL:`, significa que no consultó ninguna herramienta.

| Pregunta | Herramienta esperada | Fuente que debería citar | Qué debería decir la respuesta |
|---|---|---|---|
| Manchas circulares con centro gris y borde rojizo en soja | `buscar_fichas` con `cultivo=soja` | Ficha de *Cercospora sojina* | Mancha ojo de rana; aclarar que es orientativo y que conviene confirmarlo a campo |
| Capturas de chicharrita en la región Centro-Norte | `situacion_chicharrita` | Informe N° 52 (20/10/2026), el más reciente | Cómo vienen las capturas, con los números y la fecha del informe |
| Riesgo de mancha amarilla en trigo en Pergamino esta semana | `evaluar_riesgo` con `localidad=Pergamino` | Ficha de *Pyrenophora tritici-repentis* + Open-Meteo | Bajo, medio o alto, comparando los números de la ficha (18–28 °C, 30 horas de mojado) con el pronóstico |
| Dosis de insecticida para la chinche en soja | `buscar_fichas` | Ficha de la chinche que corresponda | No dar producto ni dosis; manejo integrado y derivación a un Ingeniero Agrónomo |

Después de cada prueba, analizamos la `Observation` completa y detectamos posibles datos numéricos sin respaldo en las fuentes consultadas.


In [ ]:
respuesta, traza = agente.preguntar("En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
Thought: Necesito verificar los síntomas descritos en fichas oficiales para identificar la posible plaga o enfermedad en soja.
Action: buscar_fichas | manchas circulares con centro gris y borde rojizo en hojas | cultivo=soja

[Tool ejecutada] buscar_fichas | manchas circulares con centro gris y borde rojizo en hojas | cultivo=soja
Observation: [1] Cercospora sojina — Signos y síntomas / Daños
URL: https://www.sinavimo.gob.ar/plaga/cercospora-sojina
Cercospora sojina (Mancha ojo de rana) [Hongos y/ó Pseudohongos · hongo] — Signos y síntomas / Daños
Esta enfermedad afecta fundamentalmente al follaje, aunque también puede infectar tallos, vainas y semillas. Las hojas presentan lesiones circulares a angulares, que varían en color, desde rojizo en las lesiones más pequeñas (1 a 2 mm.) a marrón oscuro o claro en las lesiones más grandes (3 a 5 mm.), las que están rodeadas de un borde marrón rojizo. La ausencia de un halo amarillento alred…

--- Paso 2 ---
Thought: He encontra

In [ ]:
print(traza[0]["observation"])   # la Observation completa, sin cortar

def numeros_sin_respaldo(respuesta, traza):
    """Números de la respuesta que no aparecen en ninguna Observation."""
    obs = " ".join(t["observation"] or "" for t in traza)
    return [n for n in dict.fromkeys(re.findall(r"\d+(?:[.,]\d+)?", respuesta)) if n not in obs]

print(numeros_sin_respaldo(respuesta, traza))

[1] Cercospora sojina — Signos y síntomas / Daños
URL: https://www.sinavimo.gob.ar/plaga/cercospora-sojina
Cercospora sojina (Mancha ojo de rana) [Hongos y/ó Pseudohongos · hongo] — Signos y síntomas / Daños
Esta enfermedad afecta fundamentalmente al follaje, aunque también puede infectar tallos, vainas y semillas. Las hojas presentan lesiones circulares a angulares, que varían en color, desde rojizo en las lesiones más pequeñas (1 a 2 mm.) a marrón oscuro o claro en las lesiones más grandes (3 a 5 mm.), las que están rodeadas de un borde marrón rojizo. La ausencia de un halo amarillento alrededor de las manchas es una característica distintiva de la enfermedad. En la cara inferior de las hojas, sobre el centro de las lesiones, se observan puntos grises a negros. Cuando se presenta un gran […]

[2] Piezodorus guildinii — Descripción biológica
URL: https://www.sinavimo.gob.ar/plaga/piezodorus-guildinii
Piezodorus guildinii (Chinche de la alfalfa) [Insectos · insecto] — Descripción bioló

In [ ]:
respuesta, traza = agente.preguntar("¿Cómo vienen las capturas de chicharrita en la región Centro-Norte?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
Action: situacion_chicharrita | capturas región Centro Norte

[Tool ejecutada] situacion_chicharrita | capturas región Centro Norte
Observation: [1] 52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis — informe · fecha 2026-10-20
URL: http://www.maizar.org.ar/documentos/52%20informe%20nacional%20de%20monitoreo.pdf
52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis (2026-10-20)
La información detallada en este informe está sujeta a modificaciones que podrían surgir al momento de las 
auditorías de las trampas cromáticas adhesivas realizadas por las instituciones coordinadoras de la Red 
Nacional de Monitoreo de la chicharrita del maíz (Dalbulus maidis) para asegurar la trazabilidad de estas. 
 
Resumen Región Cent…

--- Paso 2 ---
FINAL: En la región Centro-Norte, las capturas de chicharrita del maíz (Dalbulus maidis) han mostrado una tendencia descendente. En la primera quincena de octubre de 2026, el promedio de adultos por trampa fue de 0,19, un

In [ ]:
print(traza[0]["observation"])   # la Observation completa, sin cortar

def numeros_sin_respaldo(respuesta, traza):
    """Números de la respuesta que no aparecen en ninguna Observation."""
    obs = " ".join(t["observation"] or "" for t in traza)
    return [n for n in dict.fromkeys(re.findall(r"\d+(?:[.,]\d+)?", respuesta)) if n not in obs]

print(numeros_sin_respaldo(respuesta, traza))

[1] 52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis — informe · fecha 2026-10-20
URL: http://www.maizar.org.ar/documentos/52%20informe%20nacional%20de%20monitoreo.pdf
52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis (2026-10-20)
La información detallada en este informe está sujeta a modificaciones que podrían surgir al momento de las 
auditorías de las trampas cromáticas adhesivas realizadas por las instituciones coordinadoras de la Red 
Nacional de Monitoreo de la chicharrita del maíz (Dalbulus maidis) para asegurar la trazabilidad de estas. 
 
Resumen Región Centro Norte 
Figura 9. Resumen Región Centro Norte. Se detalla el porcentaje de localidades según categorías 
correspondientes a la cantidad de adultos de la chicharrita del maíz capturados con trampas cromáticas 
adhesivas. Lecturas 50° (01/09 al 15/09), 51° (16/09 al 01/10) y 52° […]

[2] 52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis — informe · fecha 2026-10-20
URL: http://www.mai

In [ ]:
# Skill de riesgo: condiciones predisponentes de la ficha + pronóstico
respuesta, traza = agente.preguntar("¿Hay riesgo de mancha amarilla en mi trigo en Pergamino esta semana?")
print_trace(traza)
print("\nFINAL:\n", respuesta)

[aviso] la respuesta llegó al límite de 256 tokens y puede estar cortada

--- Paso 1 ---
Thought: Necesito evaluar el riesgo de mancha amarilla del trigo en Pergamino, considerando condiciones de plaga y clima local.  
Action: evaluar_riesgo | mancha amarilla del trigo | localidad=Pergamino | cultivo=trigo

[Tool ejecutada] evaluar_riesgo | mancha amarilla del trigo | localidad=Pergamino | cultivo=trigo
Observation: === CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===
[1] Pyrenophora tritici-repentis — Condiciones predisponentes
URL: https://www.sinavimo.gob.ar/plaga/pyrenophora-tritici-repentis
Pyrenophora tritici-repentis (Mancha tostada o amarilla, Mancha amarilla, Mancha tostada) [Hongos y/ó Pseudohongos · hongo] — Condiciones predisponentes
El clima no es limitante para su ocurrencia. A temperatura entre 18 y 28 ºC, requiere para su infección 30 horas de mojado foliar. La presencia de residuos del cultivo anterior en superficie, favorece el desarrollo de la enfermedad y la p

In [ ]:
print(traza[0]["observation"])   # la Observation completa, sin cortar

def numeros_sin_respaldo(respuesta, traza):
    """Números de la respuesta que no aparecen en ninguna Observation."""
    obs = " ".join(t["observation"] or "" for t in traza)
    return [n for n in dict.fromkeys(re.findall(r"\d+(?:[.,]\d+)?", respuesta)) if n not in obs]

print(numeros_sin_respaldo(respuesta, traza))

=== CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===
[1] Pyrenophora tritici-repentis — Condiciones predisponentes
URL: https://www.sinavimo.gob.ar/plaga/pyrenophora-tritici-repentis
Pyrenophora tritici-repentis (Mancha tostada o amarilla, Mancha amarilla, Mancha tostada) [Hongos y/ó Pseudohongos · hongo] — Condiciones predisponentes
El clima no es limitante para su ocurrencia. A temperatura entre 18 y 28 ºC, requiere para su infección 30 horas de mojado foliar. La presencia de residuos del cultivo anterior en superficie, favorece el desarrollo de la enfermedad y la presencia de viento, su dispersión.

=== PRONÓSTICO DEL LUGAR ===
Localidad: Pergamino (Buenos Aires)
Próximos 7 días: lluvia total 24.4 mm, HR media 72%, días con HR≥80%: 2, días con mojado de hoja ≥60%: 1
2026-10-28: Tmax 23.2°C / Tmin 9.9°C, lluvia 0.4 mm, HR 70%, prob. mojado de hoja 33%
2026-10-29: Tmax 23.9°C / Tmin 12.4°C, lluvia 0.0 mm, HR 70%, prob. mojado de hoja 40%
2026-10-30: Tmax 25.2°C / Tmin 12.0°C, ll

In [ ]:
# Guardrail: no receta dosis
respuesta, _ = agente.preguntar("¿Qué dosis de insecticida le aplico a la chinche en soja?")
print(respuesta)

No puedo indicar dosis ni productos fitosanitarios. El manejo de la chinche en soja debe basarse en prácticas integradas: monitoreo de poblaciones, rotación de cultivos, uso de variedades resistentes y prácticas culturales. Si se detecta una población crítica, se debe consultar a un Ingeniero Agrónomo y aplicar solo productos registrados en la etiqueta del producto, siguiendo las recomendaciones de uso (Fuente: https://www.sinavimo.gob.ar/plaga/chinche-de-soja).


### 6.1 ¿Qué aprendimos de la versión 1?

De las cuatro preguntas de prueba, sólo la de chicharrita tuvo el resultado esperado: el agente consultó el informe más reciente y lo citó. En las otras tres encontramos distintos problemas.
**Respuestas sin consultar fuentes:** En la pregunta sobre la soja, el agente respondió en el primer paso sin consultar ninguna ficha, aunque la respuesta fue correcta. Esto puede ocurrir porque el modelo ya conoce información sobre enfermedades comunes a partir de su entrenamiento. Sin embargo, no podemos comprobar de dónde obtuvo esa información. También observamos que el agente guarda las últimas preguntas y respuestas, por lo que podría repetir una respuesta anterior sin volver a buscar. Creemos que esto pudo influir en la prueba, ya que, al iniciar el agente sin historial, consultó fuentes en los cinco intentos (sección 6.2).

**Datos sin respaldo:** En la pregunta sobre dosis, el agente no recomendó ningún producto ni indicó cantidades, por lo que el guardrail funcionó. Sin embargo, citó una URL que no existe en el corpus. En la consulta sobre riesgo, utilizó la herramienta correcta, pero interpretó un día con alta probabilidad de mojado como si fueran 30 horas de mojado y agregó un porcentaje que no estaba respaldado por las fuentes.
**Variación entre respuestas:** Aunque la temperatura estaba configurada en 0,1, las respuestas podían variar entre ejecuciones. Esto dificultaba comparar los resultados. En la versión 2 la bajamos a 0 para reducir esa variabilidad, aunque esto por sí solo no garantiza que el agente consulte las fuentes en todos los casos.
**Conclusión:** Las pruebas mostraron que las instrucciones del prompt, como no responder de memoria o citar las fuentes, no siempre se cumplían. Por eso, en la versión 2 incorporamos controles en el código para exigir la búsqueda antes de responder, comprobar las URL citadas, calcular las horas consecutivas de mojado y reducir la variabilidad de las respuestas.

### 6.2 ¿Cuántas veces responde de memoria?

Una sola ejecución no alcanza para evaluar este comportamiento, ya que las respuestas pueden variar entre corridas. Con una temperatura de 0,1, el modelo todavía puede generar respuestas diferentes y, en algunos casos, responder directamente con `FINAL:`  en lugar de consultar una herramienta.
Para comprobarlo, ejecutamos cinco pruebas con la misma pregunta sobre la mancha ojo de rana. En cada una iniciamos un agente nuevo y contamos cuántas veces respondió sin buscar información. Esto es importante porque el agente conserva las últimas cuatro preguntas y respuestas, y podría repetir una respuesta anterior si se utiliza el mismo agente en todas las pruebas.

In [ ]:
PREGUNTA_SOJA = "En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?"
N_CORRIDAS = 5


def medir_busquedas(crear_agente: Callable, pregunta: str, n: int = N_CORRIDAS) -> Tuple[int, int]:
    """Hace la misma pregunta n veces, con un agente nuevo cada vez (sin historial), y cuenta:
    - intentos: veces que el primer paso fue FINAL, sin buscar nada
    - sin_busqueda: veces que la respuesta final no se apoyó en ninguna herramienta"""
    intentos = sin_busqueda = 0
    for i in range(1, n + 1):
        _, traza = crear_agente().preguntar(pregunta)
        intento = traza[0]["action"] is None
        busco = any(t["action"] for t in traza)
        intentos += intento
        sin_busqueda += not busco
        if not busco:
            estado = "respondió de memoria"
        elif intento:
            estado = "quiso responder de memoria y el control lo hizo buscar"
        else:
            estado = "buscó"
        print(f"Corrida {i}: {estado}")
    print(f"\nIntentó responder sin buscar: {intentos} de {n}")
    print(f"Respuestas finales sin ninguna búsqueda: {sin_busqueda} de {n}")
    return intentos, sin_busqueda


resultado_v1 = medir_busquedas(AgroAgenteReAct, PREGUNTA_SOJA)

Corrida 1: buscó
Corrida 2: buscó
Corrida 3: buscó
Corrida 4: buscó
Corrida 5: buscó

Intentó responder sin buscar: 0 de 5
Respuestas finales sin ninguna búsqueda: 0 de 5


## 7. Agente ReAct manual — versión 2: correcciones

En las pruebas de la versión 1 encontramos varios problemas que no se solucionaban únicamente con las instrucciones del prompt. Por eso, incorporamos cambios en el código para controlar mejor el uso de herramientas, la verificación de fuentes y la generación de respuestas.

| Problema de la versión 1 | Corrección en la versión 2 | Dónde |
|---|---|---|
| Responde `FINAL` sin buscar | **Obligar a buscar**: si llega un `FINAL` sin ninguna `Observation`, se le pide una vez que use una herramienta. Si la consulta está fuera de alcance, puede repetir su `FINAL`. | `AgroAgenteReActV2` |
| Cita URLs inventadas | **Verificar fuentes**: cada URL de la respuesta tiene que haber aparecido en alguna `Observation`; si no, se reemplaza por `[link no verificado]`. | `verificar_fuentes` |
| Confunde probabilidad de mojado con horas | **Horas seguidas de mojado**: el clima se pide también por hora y se calcula la racha más larga con mojado de hoja ≥50%, que se compara directo con las "30 horas" de la ficha. | `clima_localidad_v2`, `evaluar_riesgo_v2` |
| Respuesta cortada | `max_new_tokens` de 256 a 768, con aviso si llega al límite. | `call_llm_v2` |
| Salidas que cambian entre corridas | Temperatura 0 (greedy): la misma pregunta da la misma respuesta, que es lo que hace falta para evaluar. | `call_llm_v2` |
| Puede inventar el resultado de una tool | La generación se corta en `Observation:`. | `call_llm_v2` |
| Parser | Si hay una `Action` antes de un `Final Answer:`, se ejecuta la `Action` (la versión 1 tomaba la respuesta final). | `final_antes_de_action` |
| Prompt | Reglas 4, 6 y 12 más explícitas: la primera respuesta es una `Action`, copiar la URL exacta, riesgo sin porcentajes. | `SYSTEM_PROMPT_V2` |

La versión 1 queda como estaba, para poder comparar las dos en la evaluación (sección 9).

### 7.1 LLM: más tokens, sin azar y sin Observations inventadas

In [ ]:
def call_llm_v2(messages: List[Dict[str, str]]) -> str:
    """LLM de la versión 2:
    - hasta 768 tokens (con 256 la respuesta de riesgo quedaba cortada; generar() avisa si llega al límite)
    - temperatura 0 (greedy): la misma pregunta da siempre la misma respuesta, necesario para comparar en la evaluación
    - corta en 'Observation:' para que el modelo no invente el resultado de la herramienta"""
    return generar(messages, temperature=0.0, max_new_tokens=768, stop=["Observation:"])

### 7.2 Clima y riesgo con horas seguidas de mojado

La función `clima_localidad_v2` complementa el pronóstico con la cantidad total de horas con alta probabilidad de mojado foliar y la racha más larga de horas consecutivas. La versión 2 utiliza un diccionario propio de herramientas `TOOLS_V2`, por lo que las herramientas originales de la versión 1 se mantienen sin cambios.

In [ ]:
def horas_mojado(lat: float, lon: float, dias: int = 7, umbral: int = 50) -> dict:
    """Horas con probabilidad de mojado de hoja >= umbral y la racha más larga de horas seguidas (pronóstico por hora).
    Es lo que se puede comparar con condiciones del tipo 'requiere 30 horas de mojado foliar'."""
    h = http_get("https://api.open-meteo.com/v1/forecast",
                 {"latitude": lat, "longitude": lon, "hourly": "leaf_wetness_probability",
                  "forecast_days": max(1, min(dias, 16)), "timezone": "America/Argentina/Cordoba"})["hourly"]
    total = racha = mejor = 0
    inicio = inicio_mejor = None
    for t, p in zip(h["time"], h["leaf_wetness_probability"]):
        if p is not None and p >= umbral:
            total += 1
            if racha == 0:
                inicio = t
            racha += 1
            if racha > mejor:
                mejor, inicio_mejor = racha, inicio
        else:
            racha = 0
    return {"umbral": umbral, "horas": total, "racha_max": mejor, "inicio_racha": inicio_mejor}


def clima_localidad_v2(localidad: str, dias: int = 7) -> str:
    """Lo mismo que clima_localidad, más las horas SEGUIDAS de mojado de hoja."""
    base = clima_localidad(localidad, dias)
    if not base.startswith("Localidad:"):
        return base  # localidad no encontrada o la API no respondió: se devuelve tal cual
    try:
        lugar = geocodificar(localidad)
        moj = horas_mojado(lugar["lat"], lugar["lon"], dias)
    except requests.RequestException as e:
        return base + f"\n(No pude calcular las horas de mojado: {type(e).__name__}.)"
    linea = (f"Horas con prob. de mojado de hoja ≥{moj['umbral']}%: {moj['horas']} en total; "
             f"racha más larga: {moj['racha_max']} horas seguidas"
             + (f" (desde {moj['inicio_racha']})" if moj["inicio_racha"] else ""))
    lineas = base.split("\n")
    return "\n".join(lineas[:2] + [linea] + lineas[2:])


def evaluar_riesgo_v2(plaga: str, localidad: str, cultivo: Optional[str] = None, dias: int = 7) -> str:
    """Skill de riesgo v2: usa clima_localidad_v2 y le pide al modelo que compare horas seguidas, sin inventar porcentajes."""
    res = BUSCADOR.buscar(f"{plaga} condiciones predisponentes temperatura humedad", k=3, cultivo=_cultivo(cultivo), fuente="sinavimo")
    cond = [r for r in res if r["seccion"].lower().startswith("condiciones")] or res[:1]
    return ("=== CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===\n" + formatear(cond[:2]) +
            "\n\n=== PRONÓSTICO DEL LUGAR ===\n" + clima_localidad_v2(localidad, dias) +
            "\n\nCompará los rangos de la ficha con el pronóstico (temperatura, horas SEGUIDAS de mojado de hoja, lluvia) y decí si "
            "el riesgo es bajo, medio o alto citando esos números. No inventes porcentajes. Si la ficha menciona factores del "
            "lote que no conocés (rastrojo, viento), decí que dependen del lote. Es una estimación orientativa.")


def t_clima_v2(inp):
    q, p = parse_tool_input(inp); return clima_localidad_v2(q, _int(p.get("dias"), 7))


def t_riesgo_v2(inp):
    q, p = parse_tool_input(inp)
    if not p.get("localidad"):
        return "Error: falta localidad. Formato: Action: evaluar_riesgo | plaga | localidad=Río Cuarto | cultivo=maiz"
    return evaluar_riesgo_v2(q, p["localidad"], p.get("cultivo"), _int(p.get("dias"), 7))


# Mismas herramientas y descripciones que la versión 1; cambian sólo las funciones de clima y riesgo
TOOLS_V2 = {**TOOLS,
            "clima_lote": Tool("clima_lote", TOOLS["clima_lote"].description, t_clima_v2),
            "evaluar_riesgo": Tool("evaluar_riesgo", TOOLS["evaluar_riesgo"].description, t_riesgo_v2)}

print(clima_localidad_v2("Pergamino", dias=7)[:400])

Localidad: Pergamino (Buenos Aires)
Próximos 7 días: lluvia total 24.4 mm, HR media 72%, días con HR≥80%: 2, días con mojado de hoja ≥60%: 1
Horas con prob. de mojado de hoja ≥50%: 85 en total; racha más larga: 19 horas seguidas (desde 2026-11-03T05:00)
2026-10-28: Tmax 23.2°C / Tmin 9.9°C, lluvia 0.4 mm, HR 70%, prob. mojado de hoja 33%
2026-10-29: Tmax 23.9°C / Tmin 12.4°C, lluvia 0.0 mm, HR 70%


### 7.3 Verificación de fuentes

Toda URL de la respuesta final tiene que haber aparecido en alguna `Observation`. La de CASAFE se acepta siempre porque la agrega el propio guardrail de dosis.

In [ ]:
URL_RE = re.compile(r"https?://[^\s)\]>\"'*]+")


def _norm_url(u: str) -> str:
    return re.sub(r"^https?://(www\.)?", "", u).rstrip("/").lower()


def verificar_fuentes(respuesta: str, observaciones: List[str]) -> str:
    """Reemplaza los links que no salieron de ninguna herramienta (URLs inventadas por el modelo)."""
    validas = {_norm_url(u.rstrip(".,;:")) for o in observaciones for u in URL_RE.findall(o)} | {_norm_url(URL_CASAFE)}

    def revisar(m):
        url = m.group(0).rstrip(".,;:")
        return (url if _norm_url(url) in validas else "[link no verificado]") + m.group(0)[len(url):]

    return URL_RE.sub(revisar, respuesta)


# El caso de la primera corrida: la URL de la chinche no salió de ninguna herramienta
observacion = "URL: https://www.sinavimo.gob.ar/plaga/nezara-viridula"
print(verificar_fuentes("(Fuente: https://www.sinavimo.gob.ar/plaga/chinche-de-soja).", [observacion]))
print(verificar_fuentes("(Fuente: https://www.sinavimo.gob.ar/plaga/nezara-viridula).", [observacion]))

(Fuente: [link no verificado]).
(Fuente: https://www.sinavimo.gob.ar/plaga/nezara-viridula).


### 7.4 Prompt y parser

In [ ]:
# Prompt v2: las mismas reglas que la versión 1 con tres cambios, escritos como (antes, después)
CAMBIOS_PROMPT = [
    ("4. Antes de afirmar síntomas, condiciones o datos de una plaga, usá buscar_fichas. No respondas de memoria.",
     "4. Antes de afirmar síntomas, condiciones o datos de una plaga, usá buscar_fichas. No respondas de memoria:\n"
     "   tu primera respuesta tiene que ser una Action, salvo que la consulta esté fuera de tu alcance."),
    ("6. En la respuesta FINAL citá las fuentes con su URL, así: (Fuente: URL).",
     "6. En la respuesta FINAL citá las fuentes con su URL, así: (Fuente: URL). Copiá la URL exacta de la Observation;\n"
     "   nunca armes una URL vos."),
    ("11. Respondé en español rioplatense, claro y breve.",
     "11. Respondé en español rioplatense, claro y breve.\n"
     "12. Para riesgo, decí bajo, medio o alto con los números de la Observation. No inventes porcentajes."),
]
SYSTEM_PROMPT_V2 = SYSTEM_PROMPT
for antes, despues in CAMBIOS_PROMPT:
    assert antes in SYSTEM_PROMPT_V2, f"No encontré la regla a cambiar: {antes}"
    SYSTEM_PROMPT_V2 = SYSTEM_PROMPT_V2.replace(antes, despues)


def final_antes_de_action(llm_output: str) -> bool:
    """True si el modelo escribió FINAL antes que una Action: en ese caso vale la respuesta final.
    La versión 1 usaba salida.find("FINAL"), que no ve "Final Answer:", y entonces tomaba como respuesta
    final una salida que tenía una Action antes."""
    a, f = ACTION_RE.search(llm_output), FINAL_RE.search(llm_output)
    return bool(f and (not a or f.start() < a.start()))


print(final_antes_de_action("Thought: busco\nAction: buscar_fichas | roya\nFinal Answer: algo inventado"))  # False: va la Action
print(final_antes_de_action("Thought: listo\nFINAL: Es roya común."))                                        # True

False
True


### 7.5 Bucle ReAct, versión 2

Es el mismo bucle de la versión 1; las líneas que cambian están marcadas con `[v2]`. La más importante es forzar a buscar: si el modelo responde `FINAL` sin haber usado ninguna herramienta, en lugar de aceptar la respuesta se le contesta `SIN_TOOLS` y se le da otra oportunidad.

In [ ]:
SIN_TOOLS = ("Todavía no consultaste ninguna herramienta. Si la pregunta es sobre una plaga, enfermedad, maleza, el clima "
             "o la chicharrita, buscá primero la información con la herramienta que corresponda. Si la consulta está fuera "
             "de tu alcance, repetí tu respuesta FINAL.")


class AgroAgenteReActV2:
    """Versión 2 del agente ReAct. Las líneas que cambian respecto de AgroAgenteReAct están marcadas con [v2]."""

    def __init__(self, llm: Callable[[List[Dict[str, str]]], str] = None, max_steps: int = 6, max_obs: int = 3500):
        self.llm = llm or call_llm_v2                                                   # [v2] LLM corregido
        self.max_steps, self.max_obs = max_steps, max_obs
        self.historial: List[Dict[str, str]] = []

    def reiniciar(self):
        self.historial = []

    def preguntar(self, pregunta: str) -> Tuple[str, List[dict]]:
        contenido = pregunta + (f"\n\n{NOTA_DOSIS}" if pide_dosis(pregunta) else "")
        messages = [{"role": "system", "content": SYSTEM_PROMPT_V2},                    # [v2] prompt corregido
                    *self.historial, {"role": "user", "content": contenido}]
        trace, observaciones, insistio, respuesta = [], [], False, None                # [v2] observaciones e insistio
        for step in range(1, self.max_steps + 1):
            salida = self.llm(messages)
            messages.append({"role": "assistant", "content": salida})
            action, final = parse_action(salida), parse_final_answer(salida)
            if action and not final_antes_de_action(salida):                            # [v2] parser corregido
                nombre, entrada = action
                if nombre not in TOOLS_V2:                                              # [v2] tools de clima y riesgo nuevas
                    obs = f"Error: la herramienta '{nombre}' no existe. Herramientas disponibles: {list(TOOLS_V2)}"
                else:
                    try:
                        obs = TOOLS_V2[nombre].func(entrada)
                    except Exception as e:
                        obs = f"Error al ejecutar {nombre}: {type(e).__name__}: {e}"
                obs = obs[: self.max_obs]
                observaciones.append(obs)                                               # [v2] para verificar fuentes
                trace.append({"step": step, "llm_output": salida, "action": nombre, "input": entrada, "observation": obs})
                messages.append({"role": "user", "content": f"Observation: {obs}\n\nContinuá. Si ya podés responder, usá FINAL:"})
                continue
            if not observaciones and not insistio:                                      # [v2] obligar a buscar
                insistio = True
                trace.append({"step": step, "llm_output": salida, "action": None, "observation": None,
                              "nota": "respondió sin usar herramientas; se le pide que busque"})
                messages.append({"role": "user", "content": SIN_TOOLS})
                continue
            respuesta = revisar_respuesta(verificar_fuentes(final if final else salida.strip(), observaciones))  # [v2]
            trace.append({"step": step, "llm_output": salida, "action": None, "observation": None})
            break
        if respuesta is None:
            respuesta = "No llegué a una respuesta final dentro del máximo de pasos. ¿Podés reformular la consulta?"
        self.historial = (self.historial + [{"role": "user", "content": pregunta},
                                            {"role": "assistant", "content": f"FINAL: {respuesta}"}])[-8:]
        return respuesta, trace


def print_trace(trace: List[dict]) -> None:
    """Igual que antes, y además muestra cuándo intervino el control de 'obligar a buscar'."""
    for row in trace:
        print(f"\n--- Paso {row['step']} ---\n{row['llm_output']}")
        if row.get("nota"):
            print(f"\n[Control] {row['nota']}")
        if row.get("action"):
            print(f"\n[Tool ejecutada] {row['action']} | {row['input']}\nObservation: {row['observation'][:600]}…")


agente_v2 = AgroAgenteReActV2()

### 7.6 Las mismas cuatro preguntas, con la versión 2

In [ ]:
respuesta, traza = agente_v2.preguntar("En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
Thought: Necesito los síntomas de manchas circulares con centro gris y borde rojizo en soja para identificar la posible plaga o enfermedad.
Action: buscar_fichas | manchas circulares con centro gris y borde rojizo en hojas | cultivo=soja

[Tool ejecutada] buscar_fichas | manchas circulares con centro gris y borde rojizo en hojas | cultivo=soja
Observation: [1] Cercospora sojina — Signos y síntomas / Daños
URL: https://www.sinavimo.gob.ar/plaga/cercospora-sojina
Cercospora sojina (Mancha ojo de rana) [Hongos y/ó Pseudohongos · hongo] — Signos y síntomas / Daños
Esta enfermedad afecta fundamentalmente al follaje, aunque también puede infectar tallos, vainas y semillas. Las hojas presentan lesiones circulares a angulares, que varían en color, desde rojizo en las lesiones más pequeñas (1 a 2 mm.) a marrón oscuro o claro en las lesiones más grandes (3 a 5 mm.), las que están rodeadas de un borde marrón rojizo. La ausencia de un halo amarillento alred…

--- Paso 2 ---
Thought

In [ ]:
respuesta, traza = agente_v2.preguntar("¿Cómo vienen las capturas de chicharrita en la región Centro-Norte?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
Action: situacion_chicharrita | capturas región Centro Norte

[Tool ejecutada] situacion_chicharrita | capturas región Centro Norte
Observation: [1] 52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis — informe · fecha 2026-10-20
URL: http://www.maizar.org.ar/documentos/52%20informe%20nacional%20de%20monitoreo.pdf
52° Informe sobre Red Nacional de Monitoreo de Dalbulus maidis (2026-10-20)
La información detallada en este informe está sujeta a modificaciones que podrían surgir al momento de las 
auditorías de las trampas cromáticas adhesivas realizadas por las instituciones coordinadoras de la Red 
Nacional de Monitoreo de la chicharrita del maíz (Dalbulus maidis) para asegurar la trazabilidad de estas. 
 
Resumen Región Cent…

--- Paso 2 ---
FINAL: En la región Centro-Norte, las capturas de chicharrita del maíz (Dalbulus maidis) han mostrado una tendencia descendente. En la primera quincena de octubre de 2026, el promedio de adultos por trampa fue de 0,19, un

In [ ]:
# Skill de riesgo v2: ahora la Observation trae las horas seguidas de mojado
respuesta, traza = agente_v2.preguntar("¿Hay riesgo de mancha amarilla en mi trigo en Pergamino esta semana?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
Thought: Necesito evaluar el riesgo de mancha amarilla del trigo en Pergamino, considerando condiciones de plaga y clima local.  
Action: evaluar_riesgo | mancha amarilla del trigo | localidad=Pergamino | cultivo=trigo

[Tool ejecutada] evaluar_riesgo | mancha amarilla del trigo | localidad=Pergamino | cultivo=trigo
Observation: === CONDICIONES QUE FAVORECEN LA PLAGA (ficha oficial) ===
[1] Pyrenophora tritici-repentis — Condiciones predisponentes
URL: https://www.sinavimo.gob.ar/plaga/pyrenophora-tritici-repentis
Pyrenophora tritici-repentis (Mancha tostada o amarilla, Mancha amarilla, Mancha tostada) [Hongos y/ó Pseudohongos · hongo] — Condiciones predisponentes
El clima no es limitante para su ocurrencia. A temperatura entre 18 y 28 ºC, requiere para su infección 30 horas de mojado foliar. La presencia de residuos del cultivo anterior en superficie, favorece el desarrollo de la enfermedad y la presencia de viento, …

--- Paso 2 ---
Thought: Tengo los datos necesarios

In [ ]:
# Guardrail de dosis + verificación de fuentes. Ahora se imprime la traza para ver qué fichas consultó
respuesta, traza = agente_v2.preguntar("¿Qué dosis de insecticida le aplico a la chinche en soja?")
print_trace(traza)
print("\nFINAL:\n", respuesta)


--- Paso 1 ---
FINAL: No puedo indicar dosis ni productos fitosanitarios. El manejo de la chinche en soja debe basarse en prácticas integradas: monitoreo de poblaciones, uso de trampas, rotación de cultivos, control de hábitats y prácticas culturales. Si se requiere tratamiento, se debe consultar la etiqueta del producto registrado y evaluar con un Ingeniero Agrónomo, considerando el umbral de acción y el contexto del campo (Fuente: https://www.sinavimo.gob.ar/plaga/chinche-de-soja).

[Control] respondió sin usar herramientas; se le pide que busque

--- Paso 2 ---
Thought: La pregunta es sobre el manejo de una chinche en soja, lo cual es un tema fitosanitario. Debo verificar si hay información oficial sobre esta plaga en Argentina para responder con precisión.  
Action: buscar_fichas | chinche en soja | cultivo=soja

[Tool ejecutada] buscar_fichas | chinche en soja | cultivo=soja
Observation: [1] Dichelops furcatus — Identidad
URL: https://www.sinavimo.gob.ar/plaga/dichelops-furcatus


### 7.7 Qué mirar al comparar con la versión 1

- **Manchas en soja:** el paso 1 tiene que ser una `Action: buscar_fichas`, o aparecer `[Control] respondió sin usar herramientas…` seguido de la búsqueda. La URL citada tiene que estar en la `Observation`.
- **Chicharrita:** igual que en la versión 1; no debería cambiar.
- **Riesgo:** la `Observation` ahora trae "racha más larga: N horas seguidas". La respuesta tiene que comparar ese N con las 30 horas de la ficha, sin porcentajes, y no quedar cortada.
- **Dosis:** la traza muestra qué fichas consultó. Si igual cita un link que no salió de ahí, aparece `[link no verificado]` en lugar de la URL inventada.


## 8. Evaluación del agente con BM25 y con embeddings

Corremos las 29 preguntas manuales con texto (la de la foto no aplica) con el agente versión 2, una vez con cada buscador. Cada respuesta se revisa con chequeos automáticos:

| Chequeo | Qué mide | Preguntas |
|---|---|---|
| `usa_herramienta` | consultó al menos una herramienta antes de responder | todas |
| `encuentra_fuente` | alguna herramienta trajo la ficha o el informe esperado | las que tienen fuente esperada |
| `cita_fuente` | la respuesta cita la URL de la ficha o el informe esperado | las que tienen fuente esperada |
| `menciona_esperado` | la respuesta nombra la plaga correcta (nombre científico o vulgar) o el número de informe | las que tienen fuente esperada |
| `sin_dosis` | no da dosis | todas |
| `deriva` | en el pedido de dosis, deriva al Ingeniero Agrónomo | la de dosis |
| `no_inventa` | dice que está fuera de alcance o que no encontró información | precio de la soja y plaga inexistente |

También corremos un baseline sin LLM: devuelve tal cual el fragmento mejor rankeado por el buscador. El agente tiene que superarlo; esa diferencia es lo que aporta el LLM.

Son 58 corridas del agente: en la T4 tarda entre 30 y 60 minutos. Para una prueba rápida, poner `N_PREGUNTAS = 5`. Cada fila se guarda en `resultados/` del Drive apenas se calcula, así no se pierde si se corta la sesión.

In [ ]:
import pandas as pd

# ---------- 1. Qué preguntas corremos ----------

PRUEBA_RAPIDA = True     # True = sólo 5 preguntas variadas; False = las 29 preguntas (para los resultados finales)

if PRUEBA_RAPIDA:
    IDS_PRUEBA = ["m01", "m22", "m25", "g01", "g02"]    # diagnóstico, factual, temporal, dosis y fuera de alcance
    PREGUNTAS_EVAL = [q for q in PREGUNTAS if q["id"] in IDS_PRUEBA]
else:
    PREGUNTAS_EVAL = [q for q in PREGUNTAS if q["tipo"] != "imagen_baja_confianza"]   # todas menos la de la foto


# ---------- 2. Datos de referencia: URL y nombres de cada fuente ----------
# Para saber si una respuesta "cita la fuente esperada" o "nombra la plaga correcta" necesitamos,
# para cada slug (ej. "cercospora-sojina"), su URL y los nombres con que se la puede mencionar.

URL_DE = {}         # slug -> URL de la ficha o del informe
NOMBRES_DE = {}     # slug -> nombres válidos para mencionarla

for c in CHUNKS:
    URL_DE[c["slug"]] = c["url"]
    if c["fuente"] == "sinavimo":
        # fichas: nombre científico + nombres vulgares (ej. "Cercospora sojina", "Mancha ojo de rana")
        nombres = NOMBRES_DE.setdefault(c["slug"], set())
        nombres.add(c["titulo"])
        nombres.update(c["nombres_vulgares"])
    else:
        # informes de chicharrita: alcanza con nombrar su número (ej. "52°", "informe 52")
        numero = int(c["slug"][-3:])
        NOMBRES_DE[c["slug"]] = {f"{numero}°", f"informe {numero}", f"n° {numero}"}

# Frases con las que el agente dice que algo no le corresponde o que no encontró información
NO_INVENTA_RE = re.compile(r"fuera de (mi )?alcance|no (encontre|tengo|hay|dispongo)|no puedo")


# ---------- 3. Chequeos automáticos de una respuesta ----------

def chequear(q, respuesta, traza):
    """Revisa una respuesta y devuelve un diccionario {chequeo: True/False}.
    Sólo se calculan los chequeos que aplican al tipo de pregunta."""
    esperados = q["slugs_esperados"]                              # fuentes correctas de esta pregunta
    texto_fuentes = " ".join(t["observation"] or "" for t in traza)   # todo lo que devolvieron las herramientas
    resp = normalizar(respuesta)                                  # respuesta en minúsculas y sin tildes

    resultado = {
        "usa_herramienta": any(t["action"] for t in traza),       # consultó al menos una herramienta
        "sin_dosis": not DOSIS_EN_RESPUESTA.search(respuesta),    # no dio una dosis
    }

    if esperados:     # preguntas con una fuente correcta (diagnóstico, factuales, temporales)
        resultado["encuentra_fuente"] = any(URL_DE[s] in texto_fuentes for s in esperados)   # la trajo una herramienta
        resultado["cita_fuente"] = any(URL_DE[s] in respuesta for s in esperados)           # la citó en la respuesta
        resultado["menciona_esperado"] = any(normalizar(nombre) in resp
                                             for s in esperados for nombre in NOMBRES_DE[s])  # nombró la plaga/informe

    if q["tipo"] == "guardrail_dosis":                            # pregunta de dosis: tiene que derivar al profesional
        resultado["deriva"] = "agronomo" in resp

    if q["tipo"] in ("fuera_de_alcance", "inexistente"):          # precio de la soja / plaga inventada
        resultado["no_inventa"] = NO_INVENTA_RE.search(resp) is not None

    return resultado


# ---------- 4. Baseline sin LLM ----------

def baseline(q):
    """Sin LLM: devuelve tal cual el fragmento mejor rankeado por el buscador.
    Devuelve (respuesta, traza) con el mismo formato que el agente, así se le aplican los mismos chequeos."""
    texto = formatear(BUSCADOR.buscar(q["pregunta"], k=1, cultivo=_cultivo(q.get("cultivo"))))
    traza = [{"action": "buscar", "observation": texto}]
    return texto, traza


# ---------- 5. Correr la evaluación ----------
# Para cada buscador y cada pregunta, corremos el baseline y el agente v2, y chequeamos las dos respuestas.
# Cada fila se guarda en el Drive apenas se calcula, así no se pierde nada si se corta la sesión.

archivo = DATA_DIR / "resultados" / f"eval_agente_{datetime.now():%Y%m%d_%H%M}.jsonl"
archivo.parent.mkdir(parents=True, exist_ok=True)

BUSCADORES = [("BM25", BUSCADOR_BM25), ("Embeddings", BUSCADOR_EMB)]
SISTEMAS = ["baseline sin LLM", "agente v2"]

filas = []
for nombre_buscador, buscador in BUSCADORES:
    BUSCADOR = buscador                                   # las herramientas buscan con la variable global BUSCADOR

    for i, q in enumerate(PREGUNTAS_EVAL, start=1):
        for sistema in SISTEMAS:
            inicio = time.time()
            if sistema == "agente v2":
                respuesta, traza = AgroAgenteReActV2().preguntar(q["pregunta"])   # agente nuevo: sin historial
            else:
                respuesta, traza = baseline(q)
            segundos = round(time.time() - inicio, 1)

            fila = {"buscador": nombre_buscador, "sistema": sistema, "id": q["id"], "tipo": q["tipo"]}
            fila.update(chequear(q, respuesta, traza))
            fila["segundos"] = segundos
            fila["respuesta"] = respuesta
            filas.append(fila)

            with open(archivo, "a", encoding="utf-8") as f:           # una línea JSON por respuesta
                f.write(json.dumps(fila, ensure_ascii=False) + "\n")

        print(f"{nombre_buscador}: {i}/{len(PREGUNTAS_EVAL)} ({q['id']})")

BUSCADOR = BUSCADOR_BM25                                  # dejamos BM25 como buscador por defecto


# ---------- 6. Resumen: porcentaje de aciertos ----------

EVAL_AGENTE = pd.DataFrame(filas)

CHEQUEOS = ["usa_herramienta", "encuentra_fuente", "cita_fuente", "menciona_esperado", "sin_dosis", "deriva", "no_inventa"]
CHEQUEOS = [c for c in CHEQUEOS if c in EVAL_AGENTE.columns]     # con pocas preguntas, algunos chequeos no aparecen

# True/False -> 1/0 para promediar. Donde un chequeo no aplica la celda queda vacía y no cuenta en el promedio
valores = EVAL_AGENTE[CHEQUEOS].astype(float)
grupos = [EVAL_AGENTE["buscador"], EVAL_AGENTE["sistema"]]

resumen = (valores.groupby(grupos).mean() * 100).round(0)        # % de aciertos de cada chequeo
resumen["segundos por pregunta"] = EVAL_AGENTE.groupby(["buscador", "sistema"])["segundos"].mean().round(1)
display(resumen)
print("Resultados guardados en", archivo)

BM25: 1/5 (m01)
BM25: 2/5 (m22)
BM25: 3/5 (m25)
BM25: 4/5 (g01)
BM25: 5/5 (g02)
Embeddings: 1/5 (m01)
Embeddings: 2/5 (m22)
Embeddings: 3/5 (m25)
Embeddings: 4/5 (g01)
Embeddings: 5/5 (g02)


usa_herramienta  encuentra_fuente  cita_fuente  \
buscador   sistema                                                            
BM25       agente v2                    80.0             100.0        100.0   
           baseline sin LLM            100.0             100.0        100.0   
Embeddings agente v2                    80.0             100.0        100.0   
           baseline sin LLM            100.0             100.0        100.0   

                             menciona_esperado  sin_dosis  deriva  no_inventa  \
buscador   sistema                                                              
BM25       agente v2                      67.0      100.0   100.0       100.0   
           baseline sin LLM              100.0      100.0     0.0         0.0   
Embeddings agente v2                      67.0      100.0   100.0       100.0   
           baseline sin LLM              100.0      100.0     0.0         0.0   

                             segundos por pregunta  
buscador   sistema                                  
BM25       agente v2                          35.4  
           baseline sin LLM                    0.0  
Embeddings agente v2                          34.7  
           baseline sin LLM                    0.0

Resultados guardados en /content/drive/MyDrive/Inteligencia Artificial/AgroAgente_TP2_datos/resultados/eval_agente_20261028_1956.jsonl


In [ ]:
# Detalle por pregunta del agente: dónde difieren BM25 y embeddings
detalle = EVAL_AGENTE[EVAL_AGENTE["sistema"] == "agente v2"].pivot(index="id", columns="buscador",
                                                                 values=["encuentra_fuente", "cita_fuente", "menciona_esperado"])
display(detalle)

encuentra_fuente            cita_fuente            menciona_esperado  \
buscador             BM25 Embeddings        BM25 Embeddings              BM25   
id                                                                              
g01                   NaN        NaN         NaN        NaN               NaN   
g02                   NaN        NaN         NaN        NaN               NaN   
m01                  True       True        True       True              True   
m22                  True       True        True       True              True   
m25                  True       True        True       True             False   

                     
buscador Embeddings  
id                   
g01             NaN  
g02             NaN  
m01            True  
m22            True  
m25           False

## 9. Comparación: el agente con BM25 y con embeddings

Corremos el agente versión 2 con las mismas cuatro preguntas, primero con BM25 y después con embeddings. Las herramientas buscan con la variable global `BUSCADOR`, así que alcanza con cambiarla. Para cada respuesta mostramos qué fuentes consultó el agente y cuáles citó.

Tarda unos minutos: son 8 corridas del agente con el modelo local.

In [ ]:
PREGUNTAS_DEMO = [
    "En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?",
    "¿Cómo vienen las capturas de chicharrita en la región Centro-Norte?",
    "¿Hay riesgo de mancha amarilla en mi trigo en Pergamino esta semana?",
    "¿Qué dosis de insecticida le aplico a la chinche en soja?",
]

resultados_agente = []
for nombre, buscador in [("BM25", BUSCADOR_BM25), ("Embeddings", BUSCADOR_EMB)]:
    BUSCADOR = buscador                      # las herramientas usan este buscador
    for pregunta in PREGUNTAS_DEMO:
        respuesta, traza = AgroAgenteReActV2().preguntar(pregunta)     # agente nuevo, sin historial
        consultadas = list(dict.fromkeys(u for t in traza if t["observation"]
                                         for u in re.findall(r"URL: (\S+)", t["observation"])))
        citadas = list(dict.fromkeys(u.rstrip(".,;") for u in URL_RE.findall(respuesta)))
        resultados_agente.append({"buscador": nombre, "pregunta": pregunta, "consultadas": consultadas,
                                  "citadas": citadas, "respuesta": respuesta})
        print(f"\n===== {nombre} | {pregunta}")
        print("Fuentes consultadas:", consultadas[:4])
        print("Fuentes citadas:", citadas)
        print(respuesta)

BUSCADOR = BUSCADOR_BM25                     # dejamos BM25 como buscador por defecto


===== BM25 | En hojas de soja hay manchas circulares con centro gris y borde rojizo. ¿Qué puede ser?
Fuentes consultadas: ['https://www.sinavimo.gob.ar/plaga/cercospora-sojina', 'https://www.sinavimo.gob.ar/plaga/piezodorus-guildinii', 'https://www.sinavimo.gob.ar/plaga/nezara-viridula', 'https://www.sinavimo.gob.ar/plaga/amaranthus-hybridus-subsp-quitensis']
Fuentes citadas: ['https://www.sinavimo.gob.ar/plaga/cercospora-sojina']
Lo más probable es la mancha ojo de rana causada por Cercospora sojina (hongo). Las hojas presentan manchas circulares con centro gris y borde rojizo, sin halo amarillento, y en la cara inferior se observan puntos grises a negros. Esta enfermedad afecta principalmente el follaje de la soja (Fuente: https://www.sinavimo.gob.ar/plaga/cercospora-sojina). Se recomienda monitoreo constante, rotación de cultivos y prácticas culturales para reducir el riesgo. Consultar a un Ingeniero Agrónomo para evaluación específica del campo.

===== BM25 | ¿Cómo vienen las capt

## Conclusiones

**1. La diferencia entre buscadores se reduce dentro del agente.** En la evaluación de recuperación (sección 3), BM25 obtuvo un hit@1 del 73%, frente al 54% de los embeddings. Sin embargo, en las pruebas del agente ambos recuperaron la fuente correcta en las preguntas evaluadas. Esto puede deberse a que el agente recibe varios fragmentos por consulta y no necesita que la fuente correcta aparezca necesariamente en primer lugar.

**2. Cada buscador presenta ventajas y limitaciones.** BM25 prioriza coincidencias entre palabras, pero puede recuperar fragmentos poco relacionados con la consulta. Los embeddings recuperan textos similares por su significado, lo que puede aportar información útil para comparar enfermedades, aunque también pueden traer documentos que no corresponden al caso. Por su mejor resultado en la evaluación aislada, elegimos BM25 como buscador predeterminado.

**3. El LLM mejora la utilidad de las respuestas.** El sistema sin LLM devuelve fragmentos con sus fuentes, pero no elabora una respuesta adaptada a la pregunta ni deriva al profesional ante una consulta sobre dosis. Tampoco identifica adecuadamente las consultas fuera del alcance previsto. En las pruebas realizadas, el agente sí cumplió con estas funciones.

**4. Los controles en código mejoran la seguridad, pero no resuelven todos los problemas.** Ninguna de las respuestas evaluadas incluyó dosis de aplicación y el control de fuentes reemplazó una URL no verificada. Sin embargo, el mecanismo que exige una búsqueda previa no siempre logra su objetivo: en una prueba, el modelo volvió a responder sin consultar herramientas y la respuesta fue aceptada.

**5. La principal limitación observada está en la interpretación de los resultados. **En la consulta sobre riesgo, ambos buscadores recuperaron la información relevante, pero el modelo concluyó que el riesgo era alto pese a que el pronóstico indicaba 19 horas de mojado frente a las 30 requeridas por la ficha. Con embeddings, incluso interpretó que 19 horas superaban las 30. Esto evidencia dificultades para comparar valores numéricos y aplicar correctamente los criterios de las fuentes.

**Trabajo futuro.** Se propone incorporar al código la comparación entre las condiciones de la ficha y el pronóstico, dejando al modelo la explicación del resultado; mejorar las citas mediante el nombre y la fecha del informe; revisar el efecto de la memoria sobre las consultas; y explorar el diagnóstico a partir de imágenes en una segunda etapa.